# AI-assisted topology synthesis: piezoelectric energy-harvester interface

Gemma 4 31B proposes interface-circuit **structures** for a piezoelectric harvester, up to three per
round. Each new structure goes through three stages:
1. **Screen** (cheap, about 1/8 of a full evaluation): the storage node is fixed at 1.8 V and the net
   power the circuit delivers is measured on the three hardest conditions, for the proposed values
   and a few automatic presets (e.g. LC-resonant switch timing). A structure that cannot deliver at
   least half of the load's need is sent back with its powers and is not tuned.
2. **Full evaluation**: 0.8 s cold-start LTspice runs on the six search conditions.
3. **Sizing**: an evolution strategy tunes all component values (up to 48 evaluations per
   structure), running while Gemma thinks about the next round.

Feedback to Gemma includes measurements, screen powers and plain-language diagnoses (e.g. "the
output never exceeds 0.7 V: check diode orientations"). When the search stops, the selected design
and the full-bridge reference are checked on four **held-out** conditions that the model never saw,
and the selected design gets a Monte Carlo robustness check (component tolerances, switch timing).

**Specification (every condition, 0.8 s cold start):** output reaches and stays at 1.8 V or more,
never exceeds 3.6 V, ripple at most 50 mV, reaches 1.8 V within 0.4 s, settled. Conditions vary
the load (150 kOhm to 2.2 MOhm), excitation (30-100 uA), frequency (80 Hz), piezo capacitance (+20 %)
and temperature (60 C). See section 1 of `report.md` once a run finishes.

**Run the cells in order:**
1. *Setup* (always).
2. *Package files*: writes `piezo_synth/` next to this notebook (always, after any update of this notebook).
3. *Settings* for the search.
4. *Smoke test* (first time, and after any change to the device models; about 2 minutes).
5. *Single run* (typically 30-60 minutes): one cluster job; `log()` follows it, `cancel(job)` stops it.
6. *Progress* (any time), *Results*, *Figures*, *Waveforms*.
7. *Experiment campaign* (for the paper): all variants x seeds as cluster jobs, then *Aggregate*,
   which writes the paper's tables and figures into `paper_generated/`.

Progress lines are tagged by phase: `[model]` waiting for Gemma, `[screen]` power screen, `[sim]`
simulating, `[eval]` checking results, `[size]` value sizing, `[heldout]` held-out check,
`[robust]` Monte Carlo, `[done]`.


## 1. Setup

In [ ]:
import os, json, time, glob
from pathlib import Path
from gpu import run, log, jobs, cancel

os.makedirs("piezo_synth", exist_ok=True)
os.makedirs("runs", exist_ok=True)
NAMESPACE = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()
LTSPICE = f"image-registry.openshift-image-registry.svc:5000/{NAMESPACE}/ltspice-cpu:latest"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")   # re-run this cell for a new run id
print("LTspice image:", LTSPICE)
print("this run will write to runs/" + RUN_ID)

## 2. Package files

The next cells write the experiment's code into `piezo_synth/` beside this notebook (cells are
collapsed; expand to read). The job reads them from there. The specification lives in
`piezo_synth/spec.py`.

In [ ]:
%%writefile piezo_synth/__init__.py
"""AI-assisted topology synthesis for piezoelectric energy-harvesting interface circuits."""
__version__ = "0.1.0"


In [ ]:
%%writefile piezo_synth/spec.py
"""The experiment specification: operating conditions, pass/fail criteria, limits, simulation settings.

Search conditions (S*) are shown to the model and used for scoring. Held-out conditions (H*) are
evaluated only on the selected design and the reference, after the search has stopped.
"""
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import asdict, dataclass
from typing import List


@dataclass(frozen=True)
class Condition:
    id: str
    split: str          # "search" | "heldout"
    label: str
    ip: float           # piezo current amplitude, A peak
    freq: float         # excitation frequency, Hz
    cp: float           # piezo capacitance, F
    rp: float           # piezo leakage resistance, ohm
    rl: float           # load resistance, ohm
    temp: float         # simulation temperature, degC


@dataclass(frozen=True)
class Criterion:
    id: str
    meas: str           # measurement name (see netlist.MEAS_NAMES, plus derived "dv")
    op: str             # ">=" | "<="
    limit: float
    scale: float        # normalises margins: margin = (meas - limit)/scale for ">="
    unit: str
    description: str


RP = 20e6

_P1_CONDITIONS: List[Condition] = [
    Condition("S1", "search", "nominal", 50e-6, 100, 25e-9, RP, 330e3, 27),
    Condition("S2", "search", "heavy load", 50e-6, 100, 25e-9, RP, 150e3, 27),
    Condition("S3", "search", "strong excitation, light load", 100e-6, 100, 25e-9, RP, 2.2e6, 27),
    Condition("S4", "search", "weak excitation", 30e-6, 100, 25e-9, RP, 1e6, 27),
    Condition("S5", "search", "low frequency", 40e-6, 80, 25e-9, RP, 470e3, 27),
    Condition("S6", "search", "Cp +20 %, hot", 50e-6, 100, 30e-9, RP, 330e3, 60),
    Condition("H1", "heldout", "weak excitation, mid load", 35e-6, 100, 25e-9, RP, 470e3, 27),
    Condition("H2", "heldout", "high frequency, Cp −15 %", 60e-6, 120, 21.25e-9, RP, 220e3, 27),
    Condition("H3", "heldout", "very strong excitation, open load, cold", 120e-6, 100, 25e-9, RP, 10e6, -20),
    Condition("H4", "heldout", "hot, heavy load", 50e-6, 100, 25e-9, RP, 200e3, 85),
]

_P1_CRITERIA: List[Criterion] = [
    Criterion("vout_min", "vavg", ">=", 1.8, 1.8, "V", "settled mean output reaches the load's minimum supply"),
    Criterion("vout_max", "vmax", "<=", 3.6, 3.6, "V", "output never exceeds the load's absolute maximum"),
    Criterion("ripple", "vpp", "<=", 0.05, 0.05, "V", "peak-to-peak ripple in the final window"),
    Criterion("startup", "tup", "<=", 0.4, 0.4, "s", "cold start: first time output reaches 1.8 V"),
    Criterion("settled", "dv", "<=", 0.02, 0.02, "V", "mean output changes little between the last two windows"),
]

LIMITS = {
    "max_components": 16,
    "max_switches": 4,
    "max_sync": 2,
    "R": (1.0, 1e8),
    "C": (1e-10, 47e-6),
    "L": (1e-5, 1e-2),
    "ton": (1e-7, 1e-3),
    "phase": (-0.25, 0.25),
    "l_rser_base": 2.0,
    "l_rser_per_h": 5000.0,
}

SIM = {"tstop": 0.8, "maxstep": 2e-6, "win": (0.7, 0.8), "win_prev": (0.6, 0.7)}

CTRL_SINK_A = 0.5e-6   # supply current charged per SYNC controller, drawn from "out"

# Fast screen (before any value tuning): the storage node is held at vstore by a source behind a
# sense resistor, and the net power delivered into it is averaged over the last avg_cycles of a
# short run. A design whose net power at 1.8 V is below the load's demand V^2/RL on some condition
# cannot hold vout >= 1.8 V there whatever its storage capacitor; alpha < 1 relaxes the test because
# the values are not tuned yet (the same idea as a relaxed DC-gain filter for unsized amplifiers).
# Robustness check of the selected design (after the search): each Monte Carlo sample draws every
# value uniformly within +/- these relative tolerances (phase: absolute, in periods, standing for
# controller timing error), and is evaluated on all conditions.
TOLERANCES = {"R": 0.05, "C": 0.2, "L": 0.1, "ton": 0.05, "phase": 0.02}

SCREEN = {"vstore": 1.8, "rsense": 100.0, "cycles": 10, "avg_cycles": 4, "alpha": 0.5,
          "conditions": ("S2", "S4", "S6"), "presets": 4, "spread": 3.0}


def _criteria(vmin: float, vmax: float, ripple: float = 0.05, startup: float = 0.4,
              settled: float = 0.02) -> List[Criterion]:
    return [
        Criterion("vout_min", "vavg", ">=", vmin, vmin, "V", "settled mean output reaches the load's minimum supply"),
        Criterion("vout_max", "vmax", "<=", vmax, vmax, "V", "output never exceeds the load's absolute maximum"),
        Criterion("ripple", "vpp", "<=", ripple, ripple, "V", "peak-to-peak ripple in the final window"),
        Criterion("startup", "tup", "<=", startup, startup, "s",
                  f"cold start: first time output reaches {vmin:g} V"),
        Criterion("settled", "dv", "<=", settled, settled, "V",
                  "mean output changes little between the last two windows"),
    ]


def _c(cid, label, ip_ua, freq, cp_nf, rl, temp=27.0):
    return Condition(cid, "search" if cid.startswith("S") else "heldout", label, ip_ua * 1e-6, freq, cp_nf * 1e-9,
                     RP, rl, temp)


@dataclass(frozen=True)
class Spec:
    name: str
    title: str
    hardness: str            # what makes it hard; for the paper and the report, never shown to the model
    conditions: List[Condition]
    criteria: List[Criterion]


SPECS = {
    "P1": Spec("P1", "Sensor supply 1.8-3.6 V", "the full bridge lacks power at weak excitation and heavy load, "
               "and overcharges at strong excitation", _P1_CONDITIONS, _P1_CRITERIA),
    # A 4.5-5.5 V rail: on most conditions the open-circuit voltage is below the window plus two diode drops,
    # so a bridge alone never conducts; strong excitation exceeds 5.5 V, and the only zener breaks at 3.3 V.
    "P2": Spec("P2", "High-voltage rail 4.5-5.5 V", "open-circuit voltage below the output window on most "
               "conditions; the clamp must be composed (the only zener is 3.3 V)", [
        _c("S1", "nominal", 60, 100, 25, 2.2e6),
        _c("S2", "heavy load", 60, 100, 25, 1e6),
        _c("S3", "strong excitation, light load", 120, 100, 25, 10e6),
        _c("S4", "weak excitation", 40, 100, 25, 4.7e6),
        _c("S5", "low frequency", 50, 80, 25, 3.3e6),
        _c("S6", "Cp +20 %, hot", 60, 100, 30, 2.2e6, 60),
        _c("H1", "weak excitation, mid load", 45, 100, 25, 3.3e6),
        _c("H2", "high frequency, Cp -15 %", 70, 120, 21.25, 1.5e6),
        _c("H3", "very strong excitation, open load, cold", 140, 100, 25, 22e6, -20),
        _c("H4", "hot, heavy load", 60, 100, 25, 1.5e6, 85),
    ], _criteria(4.5, 5.5)),
    # Very weak excitation: the open-circuit voltage is below half of 1.8 V, and one synchronous controller's
    # supply current costs more power than the load needs, so the switched interfaces of P1 lose energy.
    "P3": Spec("P3", "Very weak excitation, 1.8-3.6 V", "open-circuit voltage below half the minimum output; a "
               "synchronous controller costs more than the load", [
        _c("S1", "nominal (weak)", 12, 100, 25, 10e6),
        _c("S2", "heavy load", 12, 100, 25, 6.8e6),
        _c("S3", "strong excitation, light load", 40, 100, 25, 22e6),
        _c("S4", "weakest excitation", 8, 100, 25, 22e6),
        _c("S5", "low frequency", 10, 80, 25, 10e6),
        _c("S6", "Cp +20 %, hot", 12, 100, 30, 10e6, 60),
        _c("H1", "weak, mid load", 10, 100, 25, 15e6),
        _c("H2", "high frequency, Cp -15 %", 15, 120, 21.25, 6.8e6),
        _c("H3", "strong excitation, open load, cold", 50, 100, 25, 100e6, -20),
        _c("H4", "warm, heavy load", 12, 100, 25, 6.8e6, 70),
    ], _criteria(1.8, 3.6)),
    # Wide band: 50-200 Hz with the source current proportional to frequency, Cp 17.5-32.5 nF, and the hot,
    # heavy-load corner that P1 kept for the held-out set is now a search condition.
    "P4": Spec("P4", "Wide band 50-200 Hz, Cp +/-30 %", "timing and resonance must hold over a 4x frequency "
               "and 1.9x capacitance range, including 85 C at heavy load", [
        _c("S1", "nominal", 50, 100, 25, 330e3),
        _c("S2", "low frequency, mid load", 25, 50, 25, 680e3),
        _c("S3", "high frequency, small Cp, light load", 100, 200, 17.5, 4.7e6),
        _c("S4", "low frequency, large Cp", 25, 50, 32.5, 1e6),
        _c("S5", "high frequency, large Cp, heavy load", 100, 200, 32.5, 150e3),
        _c("S6", "hot, heavy load", 50, 100, 25, 200e3, 85),
        _c("H1", "75 Hz, Cp +12 %", 37.5, 75, 28, 470e3),
        _c("H2", "150 Hz, Cp -20 %", 75, 150, 20, 220e3),
        _c("H3", "200 Hz, strong, open load, cold", 120, 200, 17.5, 10e6, -20),
        _c("H4", "60 Hz, Cp +20 %, warm", 30, 60, 30, 470e3, 70),
    ], _criteria(1.8, 3.6)),
    # P1's conditions with a tight supply: small ripple needs a large storage capacitance, a fast start a
    # small one.
    "P5": Spec("P5", "Fast start, low ripple, 1.8-3.6 V", "ripple <= 20 mV needs storage capacitance that a "
               "start within 0.15 s cannot charge", list(_P1_CONDITIONS),
               _criteria(1.8, 3.6, ripple=0.02, startup=0.15, settled=0.01)),
}

ACTIVE = "P1"
CONDITIONS: List[Condition] = list(_P1_CONDITIONS)
CRITERIA: List[Criterion] = list(_P1_CRITERIA)


def use(name: str) -> None:
    """Make `name` the active specification. The module-level lists and SCREEN are changed in place, so
    modules that imported them see the new spec."""
    global ACTIVE
    sp = SPECS[name]
    CONDITIONS[:] = sp.conditions
    CRITERIA[:] = sp.criteria
    SCREEN["vstore"] = vmin()
    ACTIVE = name


@contextmanager
def using(name: str):
    prev = ACTIVE
    use(name)
    try:
        yield SPECS[name]
    finally:
        use(prev)


def _limit(cid: str) -> float:
    return next(c.limit for c in CRITERIA if c.id == cid)


def vmin() -> float:
    return _limit("vout_min")


def vmax() -> float:
    return _limit("vout_max")


def limit(cid: str) -> float:
    return _limit(cid)


def search_conditions() -> List[Condition]:
    return [c for c in CONDITIONS if c.split == "search"]


def heldout_conditions() -> List[Condition]:
    return [c for c in CONDITIONS if c.split == "heldout"]


def condition(cid: str) -> Condition:
    for c in CONDITIONS:
        if c.id == cid:
            return c
    raise KeyError(cid)


def _jsonable(v):
    return list(v) if isinstance(v, tuple) else v


def spec_dict() -> dict:
    return {
        "name": ACTIVE,
        "title": SPECS[ACTIVE].title,
        "conditions": [asdict(c) for c in CONDITIONS],
        "criteria": [asdict(c) for c in CRITERIA],
        "limits": {k: _jsonable(v) for k, v in LIMITS.items()},
        "sim": {k: _jsonable(v) for k, v in SIM.items()},
        "ctrl_sink_a": CTRL_SINK_A,
        "screen": {k: _jsonable(v) for k, v in SCREEN.items()},
        "tolerances": dict(TOLERANCES),
    }


In [ ]:
%%writefile piezo_synth/circuit.py
"""Circuit representation proposed by the model: parse, validate, canonicalise, hash, edit values.

A circuit is a list of components over a fixed vocabulary, connected through named nodes.
Reserved nodes: "0" (ground), "pzp"/"pzn" (piezo terminals), "out" (storage/load node).
The test bench (piezo model, load, controller supply sinks) is added by netlist.py.
"""
from __future__ import annotations

import copy
import hashlib
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

from .spec import LIMITS

ALLOWED_TYPES = ("R", "C", "L", "D", "S", "SYNC")
DIODE_MODELS = ("DSCH", "DSI", "DZ33")
RESERVED_NODES = ("0", "pzp", "pzn", "out")
MOVES = ("new_topology", "modify_topology", "resize")
EDGES = ("both", "pos", "neg")
TWO_TERMINAL = ("R", "C", "L", "D", "S")
VALUED = ("R", "C", "L")
# connections the harness itself makes to each reserved node (piezo model, load)
HARNESS_CONNECTIONS = {"pzp": 1, "pzn": 1, "out": 1}

_ID_RE = re.compile(r"^[A-Za-z][A-Za-z0-9_]{0,7}$")
_NODE_RE = re.compile(r"^[a-z][a-z0-9_]{0,11}$")
_SUFFIX = {"f": 1e-15, "p": 1e-12, "n": 1e-9, "u": 1e-6, "µ": 1e-6, "m": 1e-3,
           "k": 1e3, "meg": 1e6, "g": 1e9, "t": 1e12}


class CircuitError(ValueError):
    def __init__(self, problems: List[str]):
        super().__init__("; ".join(problems))
        self.problems = list(problems)


@dataclass
class Component:
    id: str
    type: str
    nodes: tuple = ()
    value: Optional[float] = None
    model: Optional[str] = None
    ctrl: Optional[str] = None
    ton: Optional[float] = None
    phase: Optional[float] = None
    edges: Optional[str] = None


@dataclass
class Circuit:
    name: str
    components: List[Component] = field(default_factory=list)
    move: str = "new_topology"
    rationale: str = ""
    done: bool = False


def parse_value(v) -> Optional[float]:
    """A number, or a SPICE-style string such as '2.2u', '4.7meg', '10k'. None if unparseable."""
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if not isinstance(v, str):
        return None
    s = v.strip().lower().replace(" ", "")
    m =re.match(r"^([-+]?(?:\d+\.?\d*|\.\d+)(?:e[-+]?\d+)?)(meg|[fpnuµmkgt])?[a-z]*$", s)
    if not m:
        return None
    return float(m.group(1)) * (_SUFFIX[m.group(2)] if m.group(2) else 1.0)


# ---------------------------------------------------------------- parsing and validation

def parse_circuit(obj: dict) -> Circuit:
    """Build a Circuit from the model's JSON object; raise CircuitError listing every problem."""
    problems: List[str] = []
    if not isinstance(obj, dict):
        raise CircuitError(["the reply is not a JSON object"])
    comps_in = obj.get("components")
    if not isinstance(comps_in, list) or not comps_in:
        raise CircuitError(["'components' must be a non-empty list"])
    comps: List[Component] = []
    for k, raw in enumerate(comps_in):
        if not isinstance(raw, dict):
            problems.append(f"component #{k + 1} is not an object")
            continue
        cid = str(raw.get("id", f"#{k + 1}"))
        ctype = str(raw.get("type", "")).upper()
        nodes = raw.get("nodes", [])
        if not isinstance(nodes, list):
            problems.append(f"{cid}: 'nodes' must be a list")
            nodes = []
        num = {}
        for f in ("value", "ton", "phase"):
            if raw.get(f) is not None:
                num[f] = parse_value(raw[f])
                if num[f] is None:
                    problems.append(f"{cid}: {f}={raw[f]!r} is not a number")
        comps.append(Component(
            id=cid, type=ctype, nodes=tuple(str(n) for n in nodes),
            value=num.get("value"), model=(str(raw["model"]).upper() if raw.get("model") else None),
            ctrl=(str(raw["ctrl"]) if raw.get("ctrl") else None),
            ton=num.get("ton"), phase=num.get("phase"),
            edges=(str(raw["edges"]).lower() if raw.get("edges") else None)))
    c = Circuit(name=str(obj.get("name", "unnamed"))[:80], components=comps,
                move=str(obj.get("move", "new_topology")), rationale=str(obj.get("rationale", "")),
                done=bool(obj.get("done", False)))
    problems += validate(c)
    if problems:
        raise CircuitError(problems)
    return c


def _in(lo_hi, v) -> bool:
    return v is not None and lo_hi[0] <= v <= lo_hi[1]


def validate(c: Circuit) -> List[str]:
    """Every rule violation as a human-readable line naming the component."""
    p: List[str] = []
    if c.move not in MOVES:
        p.append(f"move={c.move!r} must be one of {', '.join(MOVES)}")
    ids = [x.id for x in c.components]
    for dup in sorted({i for i in ids if ids.count(i) > 1}):
        p.append(f"{dup}: duplicate id")
    syncs = {x.id for x in c.components if x.type == "SYNC"}
    used_syncs = set()
    for x in c.components:
        if not _ID_RE.match(x.id):
            p.append(f"{x.id}: id must be a letter followed by up to 7 letters/digits/_")
        if x.type not in ALLOWED_TYPES:
            p.append(f"{x.id}: type {x.type!r} is not allowed (use {', '.join(ALLOWED_TYPES)})")
            continue
        if x.type in TWO_TERMINAL:
            if len(x.nodes) != 2:
                p.append(f"{x.id}: needs exactly 2 nodes, got {len(x.nodes)}")
            elif x.nodes[0] == x.nodes[1]:
                p.append(f"{x.id}: both terminals on node {x.nodes[0]}")
            for n in x.nodes:
                if n != "0" and not _NODE_RE.match(n):
                    p.append(f"{x.id}: node name {n!r} must be lower-case, start with a letter, max 12 chars, or '0'")
        elif x.nodes:
            p.append(f"{x.id}: SYNC takes no nodes")
        if x.type in VALUED and not _in(LIMITS[x.type], x.value):
            lo, hi = LIMITS[x.type]
            p.append(f"{x.id}: value {x.value} outside [{lo:g}, {hi:g}]")
        if x.type == "D" and x.model not in DIODE_MODELS:
            p.append(f"{x.id}: diode model must be one of {', '.join(DIODE_MODELS)}")
        if x.type == "D" and tuple(x.nodes) == ("out", "0"):
            p.append(f'{x.id}: nodes ["out", "0"] put the anode on out, so the diode is forward-biased, holds out '
                     f'near 0.3-0.7 V and drains the storage. A zener clamp on out needs nodes ["0", "out"] '
                     f'(cathode on out)')
        if x.type == "S":
            if x.ctrl not in syncs:
                p.append(f"{x.id}: ctrl must name a SYNC component (got {x.ctrl!r})")
            else:
                used_syncs.add(x.ctrl)
        if x.type == "SYNC":
            if not _in(LIMITS["ton"], x.ton):
                p.append(f"{x.id}: ton {x.ton} outside [{LIMITS['ton'][0]:g}, {LIMITS['ton'][1]:g}] s")
            if not _in(LIMITS["phase"], 0.0 if x.phase is None else x.phase):
                p.append(f"{x.id}: phase {x.phase} outside [{LIMITS['phase'][0]}, {LIMITS['phase'][1]}] periods")
            if (x.edges or "both") not in EDGES:
                p.append(f"{x.id}: edges must be one of {', '.join(EDGES)}")
    for s in sorted(syncs - used_syncs):
        p.append(f"{s}: SYNC controller is not used by any switch")
    # connectivity
    count: Dict[str, int] = dict(HARNESS_CONNECTIONS)
    for x in c.components:
        if x.type in TWO_TERMINAL and len(x.nodes) == 2:
            for n in x.nodes:
                count[n] = count.get(n, 0) + 1
    for r in ("pzp", "pzn", "out"):
        if count.get(r, 0) <= HARNESS_CONNECTIONS[r]:
            p.append(f"node {r} is not connected to any component")
    for n, k in sorted(count.items()):
        if n not in RESERVED_NODES and k < 2:
            p.append(f"node {n} has only one connection (dangling)")
    if not _connected(c, "out", "pzp"):
        p.append("node out is not connected to the piezo (pzp) through the circuit")
    n_sw = sum(x.type == "S" for x in c.components)
    if len(c.components) > LIMITS["max_components"]:
        p.append(f"{len(c.components)} components; at most {LIMITS['max_components']} allowed")
    if n_sw > LIMITS["max_switches"]:
        p.append(f"{n_sw} switches; at most {LIMITS['max_switches']} allowed")
    if len(syncs) > LIMITS["max_sync"]:
        p.append(f"{len(syncs)} SYNC controllers; at most {LIMITS['max_sync']} allowed")
    return p


def _connected(c: Circuit, a: str, b: str) -> bool:
    adj: Dict[str, set] = {"pzp": {"pzn"}, "pzn": {"pzp"}}
    for x in c.components:
        if x.type in TWO_TERMINAL and len(x.nodes) == 2:
            u, v = x.nodes
            adj.setdefault(u, set()).add(v)
            adj.setdefault(v, set()).add(u)
    seen, stack = {a}, [a]
    while stack:
        for nb in adj.get(stack.pop(), ()):
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    return b in seen


# ---------------------------------------------------------------- canonical form and hashes

def _h(s: str) -> str:
    return hashlib.sha1(s.encode()).hexdigest()[:16]


def _pins(x: Component) -> List[Tuple[str, str]]:
    """(node, role) pairs; diodes are polar (anode/cathode), others are not."""
    if x.type == "D":
        return [(x.nodes[0], "a"), (x.nodes[1], "k")]
    return [(n, "x") for n in x.nodes]


def canonical(c: Circuit) -> Circuit:
    """Same circuit with internal nodes renamed n1..nk and ids renamed by type, in a form that does
    not depend on the names or order the model chose."""
    comps = [copy.copy(x) for x in c.components]
    nodes = sorted({n for x in comps for n in x.nodes})
    label = {n: (n if n in RESERVED_NODES else "int") for n in nodes}
    for _ in range(3):
        new = {}
        for n in nodes:
            ctx = []
            for x in comps:
                pins = _pins(x) if x.type in TWO_TERMINAL else []
                for node, role in pins:
                    if node == n:
                        others = [label[o] for o, _r in pins if o != n]
                        ctx.append(f"{x.type}/{x.model}/{role}/{','.join(others)}")
            new[n] = _h(label[n] + "|" + ";".join(sorted(ctx)))
        label = {n: (n if n in RESERVED_NODES else new[n]) for n in nodes}

    by_id = {x.id: x for x in comps}

    def sig(x: Component) -> str:
        if x.type == "SYNC":
            sw = sorted(sig(s) for s in comps if s.type == "S" and s.ctrl == x.id)
            return f"SYNC/{x.edges or 'both'}/{'+'.join(sw)}"
        pins = _pins(x)
        labs = [f"{label[n]}{r}" for n, r in pins]
        if x.type != "D":
            labs.sort()
        extra = ""
        if x.type == "S" and x.ctrl in by_id:
            extra = "/" + (by_id[x.ctrl].edges or "both")
        return f"{x.type}/{x.model}/{','.join(labs)}{extra}"

    order = sorted(range(len(comps)), key=lambda i: (sig(comps[i]), i))
    rename: Dict[str, str] = {}
    k = 0
    for i in order:
        x = comps[i]
        for n in (x.nodes if x.type == "D" else sorted(x.nodes, key=lambda m: label[m])):
            if n not in RESERVED_NODES and n not in rename:
                k += 1
                rename[n] = f"n{k}"
    out: List[Component] = []
    counters: Dict[str, int] = {}
    id_map: Dict[str, str] = {}
    for i in order:
        x = comps[i]
        prefix = "G" if x.type == "SYNC" else x.type
        counters[prefix] = counters.get(prefix, 0) + 1
        id_map[x.id] = f"{prefix}{counters[prefix]}"
    for i in order:
        x = copy.copy(comps[i])
        nn = tuple(rename.get(n, n) for n in x.nodes)
        if x.type in TWO_TERMINAL and x.type != "D":
            nn = tuple(sorted(nn))
        x.nodes = nn
        x.id = id_map[x.id]
        if x.ctrl:
            x.ctrl = id_map.get(x.ctrl, x.ctrl)
        if x.type == "SYNC":
            x.edges = x.edges or "both"
            x.phase = 0.0 if x.phase is None else x.phase
        out.append(x)
    return Circuit(name=c.name, components=out, move=c.move, rationale=c.rationale, done=c.done)


def _struct_line(x: Component) -> str:
    return f"{x.id} {x.type} {' '.join(x.nodes)} m={x.model} ctrl={x.ctrl} e={x.edges}"


def _value_line(x: Component) -> str:
    vals = [f"{v:.3g}" if v is not None else "-" for v in (x.value, x.ton, x.phase)]
    return " ".join(vals)


def topology_hash(c: Circuit) -> str:
    cc = canonical(c)
    return hashlib.sha256("\n".join(_struct_line(x) for x in cc.components).encode()).hexdigest()[:12]


def design_hash(c: Circuit) -> str:
    cc = canonical(c)
    text = "\n".join(_struct_line(x) + " | " + _value_line(x) for x in cc.components)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# ---------------------------------------------------------------- serialisation and value edits

def to_dict(c: Circuit) -> dict:
    comps = []
    for x in c.components:
        d = {"id": x.id, "type": x.type}
        if x.type in TWO_TERMINAL:
            d["nodes"] = list(x.nodes)
        for f in ("value", "model", "ctrl", "ton", "phase", "edges"):
            v = getattr(x, f)
            if v is not None:
                d[f] = v
        comps.append(d)
    return {"move": c.move, "name": c.name, "rationale": c.rationale, "done": c.done, "components": comps}


def numeric_params(c: Circuit) -> List[Tuple[str, str]]:
    out = []
    for x in c.components:
        if x.type in VALUED:
            out.append((x.id, "value"))
        elif x.type == "SYNC":
            out += [(x.id, "ton"), (x.id, "phase")]
    return out


def _find(c: Circuit, cid: str) -> Component:
    for x in c.components:
        if x.id == cid:
            return x
    raise KeyError(cid)


def get_param(c: Circuit, cid: str, fld: str) -> float:
    v = getattr(_find(c, cid), fld)
    return 0.0 if v is None else float(v)


def with_param(c: Circuit, cid: str, fld: str, value: float) -> Circuit:
    """A copy of c with one numeric field changed, clamped to the allowed range."""
    new = copy.deepcopy(c)
    x = _find(new, cid)
    lo, hi = LIMITS[x.type] if fld == "value" else LIMITS[fld]
    setattr(x, fld, min(max(float(value), lo), hi))
    return new


def describe(c: Circuit) -> str:
    """One line per component, SPICE-like, for prompts and reports."""
    lines = []
    for x in c.components:
        if x.type == "SYNC":
            lines.append(f"{x.id} SYNC ton={x.ton:.3g}s phase={x.phase or 0:.3g} edges={x.edges or 'both'}")
        elif x.type == "S":
            lines.append(f"{x.id} S {x.nodes[0]} {x.nodes[1]} ctrl={x.ctrl}")
        elif x.type == "D":
            lines.append(f"{x.id} D {x.nodes[0]} {x.nodes[1]} {x.model}")
        else:
            lines.append(f"{x.id} {x.type} {x.nodes[0]} {x.nodes[1]} {x.value:.3g}")
    return "\n".join(lines)


In [ ]:
%%writefile piezo_synth/netlist.py
"""LTspice netlist for one circuit under one operating condition, plus the reference and witness circuits.

The harness owns everything except the proposed components: the piezo model, the load, controller
supply sinks, 1 GOhm bleed resistors on internal nodes and the piezo terminals (to avoid floating-node errors), device
models, the 0.8 s cold-start transient and the measurements.
"""
from __future__ import annotations

from typing import Tuple

from .circuit import Circuit, Component, canonical
from . import spec
from .spec import CTRL_SINK_A, LIMITS, SCREEN, SIM, Condition

MODELS_TEXT = """.model DSCH D(Is=5e-8 Rs=1 N=1.05 Cjo=10p BV=30 IBV=1u EG=0.69 XTI=2)
.model DSI D(Is=2.5n Rs=0.6 N=1.75 Cjo=4p BV=75 IBV=1u)
.model DZ33 D(Is=1e-12 Rs=5 N=1.5 BV=3.3 IBV=1u)
.model SWI SW(Ron=2 Roff=1G Vt=0.5 Vh=0)"""

MEAS_NAMES = ("vavg", "vavg_prev", "vpp", "vmax", "tup", "pload", "pext")
SCREEN_MEAS_NAMES = ("isn", "vout")


def fmt(v: float) -> str:
    return "%.6g" % v


def sync_timing(ton: float, phase: float, edges: str, freq: float) -> Tuple[float, float]:
    """(t0, period) of the controller's pulses. The piezo current is Ip*sin(2*pi*f*t), so the
    open-circuit piezo voltage peaks positive at t = 1/(2f) + k/f and negative at t = k/f.
    phase shifts the pulses by that fraction of the excitation period."""
    shift = (phase or 0.0) / freq
    if edges == "pos":
        period = 1.0 / freq
        t0 = (0.5 / freq + shift) % period
    elif edges == "neg":
        period = 1.0 / freq
        t0 = shift % period
    else:
        period = 0.5 / freq
        t0 = shift % period
    if abs(t0 - period) < 1e-15 or abs(t0) < 1e-15:
        t0 = 0.0
    return t0, period


def _component_line(x: Component) -> str:
    a, b = x.nodes if x.nodes else ("", "")
    if x.type in ("R", "C"):
        return f"{x.id} {a} {b} {fmt(x.value)}"
    if x.type == "L":
        rser = LIMITS["l_rser_base"] + LIMITS["l_rser_per_h"] * x.value
        return f"{x.id} {a} {b} {fmt(x.value)} Rser={fmt(rser)}"
    if x.type == "D":
        return f"{x.id} {a} {b} {x.model}"
    if x.type == "S":
        return f"{x.id} {a} {b} {x.ctrl}_g 0 SWI"
    raise ValueError(x.type)


def _bench(cc: Circuit, cond: Condition, title: str, load_lines) -> list:
    """Piezo, load lines, controller sinks, proposed components, pulse sources, bleeds, models, .temp."""
    lines = [f"* {title or cc.name} | condition {cond.id}",
             f"Ipz pzn pzp SINE(0 {fmt(cond.ip)} {fmt(cond.freq)})",
             f"Cpz pzp pzn {fmt(cond.cp)}",
             f"Rpz pzp pzn {fmt(cond.rp)}",
             *load_lines]
    syncs = [x for x in cc.components if x.type == "SYNC"]
    for k, _ in enumerate(syncs, 1):
        lines.append(f"Ictl{k} out 0 {fmt(CTRL_SINK_A)}")
    for x in cc.components:
        if x.type != "SYNC":
            lines.append(_component_line(x))
    for x in syncs:
        t0, period = sync_timing(x.ton, x.phase, x.edges or "both", cond.freq)
        lines.append(f"V{x.id} {x.id}_g 0 PULSE(0 1 {fmt(t0)} 100n 100n {fmt(x.ton)} {fmt(period)})")
    # bleed resistors: internal nodes, and the piezo terminals (a design may couple them only through capacitors)
    internal = ["pzp", "pzn"] + sorted({n for x in cc.components for n in x.nodes} - {"0", "pzp", "pzn", "out"})
    for n in internal:
        lines.append(f"Rbleed_{n} {n} 0 1G")
    lines.append(MODELS_TEXT)
    lines.append(f".temp {fmt(cond.temp)}")
    return lines


def build_screen_netlist(c: Circuit, cond: Condition, title: str = "") -> str:
    """Short run with out held at SCREEN['vstore'] behind a sense resistor (see spec.SCREEN)."""
    cc = canonical(c)
    vst, rs = SCREEN["vstore"], SCREEN["rsense"]
    tstop = SCREEN["cycles"] / cond.freq
    w0 = tstop - SCREEN["avg_cycles"] / cond.freq
    lines = _bench(cc, cond, title, [f"Rsense out vsn {fmt(rs)}", f"Vst vsn 0 {fmt(vst)}"])
    lines += [f".tran 0 {fmt(tstop)} 0 {fmt(SIM['maxstep'])}",
              ".save V(out) V(vsn)",
              f".meas TRAN isn AVG V(out,vsn) FROM={fmt(w0)} TO={fmt(tstop)}",
              f".meas TRAN vout AVG V(out) FROM={fmt(w0)} TO={fmt(tstop)}",
              ".end"]
    return "\n".join(lines) + "\n"


def build_netlist(c: Circuit, cond: Condition, title: str = "", save_all: bool = False) -> str:
    cc = canonical(c)
    w0, w1 = SIM["win"]
    p0, p1 = SIM["win_prev"]
    tstop = SIM["tstop"]
    lines = _bench(cc, cond, title, [f"RL out 0 {fmt(cond.rl)}"])
    lines.append(f".tran 0 {fmt(tstop)} 0 {fmt(SIM['maxstep'])}")
    if not save_all:
        lines.append(".save V(out) V(pzp) V(pzn) I(Ipz)")
    lines += [
        f".meas TRAN vavg AVG V(out) FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN vavg_prev AVG V(out) FROM={fmt(p0)} TO={fmt(p1)}",
        f".meas TRAN vpp PP V(out) FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN vmax MAX V(out) FROM=0 TO={fmt(tstop)}",
        f".meas TRAN tup FIND time WHEN V(out)={fmt(spec.vmin())} RISE=1",
        f".meas TRAN pload AVG V(out)*V(out)/{fmt(cond.rl)} FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN pext AVG V(pzp,pzn)*I(Ipz) FROM={fmt(w0)} TO={fmt(w1)}",
        ".end",
    ]
    return "\n".join(lines) + "\n"


def _bridge() -> list:
    return [Component("D1", "D", ("pzp", "out"), model="DSCH"),
            Component("D2", "D", ("pzn", "out"), model="DSCH"),
            Component("D3", "D", ("0", "pzp"), model="DSCH"),
            Component("D4", "D", ("0", "pzn"), model="DSCH"),
            Component("C1", "C", ("out", "0"), value=2.2e-6)]


def reference_circuit() -> Circuit:
    """Full-bridge rectifier with a 2.2 uF storage capacitor: the standard passive interface."""
    return Circuit(name="FBR reference (4x Schottky, 2.2 uF)", components=_bridge(),
                   rationale="standard full-bridge rectifier")


def witness_circuit() -> Circuit:
    """Parallel SSHI + bridge + 3.3 V zener clamp. Used only by the smoke test to show the spec is
    reachable in principle; never shown to the model."""
    comps = _bridge() + [
        Component("L1", "L", ("pzp", "n1"), value=4.7e-3),
        Component("S1", "S", ("n1", "pzn"), ctrl="G1"),
        Component("G1", "SYNC", (), ton=34e-6, phase=0.0, edges="both"),
        Component("D5", "D", ("0", "out"), model="DZ33"),
    ]
    return Circuit(name="witness: p-SSHI + bridge + zener clamp", components=comps)


def _stacked_clamp_witness() -> Circuit:
    """P2: parallel SSHI + bridge, clamped near 5 V by the 3.3 V zener in series with three silicon diodes."""
    comps = _bridge() + [
        Component("L1", "L", ("pzp", "n1"), value=4.7e-3),
        Component("S1", "S", ("n1", "pzn"), ctrl="G1"),
        Component("G1", "SYNC", (), ton=34e-6, phase=0.0, edges="both"),
        Component("D5", "D", ("z1", "out"), model="DZ33"),
        Component("D6", "D", ("z1", "z2"), model="DSI"),
        Component("D7", "D", ("z2", "z3"), model="DSI"),
        Component("D8", "D", ("z3", "0"), model="DSI"),
    ]
    return Circuit(name="witness: p-SSHI + bridge + stacked clamp (3.3 V zener + 3 diodes)", components=comps)


def _multiplier_witness() -> Circuit:
    """P3: passive two-stage Cockcroft-Walton multiplier (no controller to supply) with a 3.3 V clamp."""
    comps = [
        Component("R1", "R", ("pzn", "0"), value=1.0),
        Component("C1", "C", ("pzp", "n1"), value=1e-6),
        Component("D1", "D", ("0", "n1"), model="DSCH"),
        Component("D2", "D", ("n1", "n2"), model="DSCH"),
        Component("C2", "C", ("n2", "0"), value=1e-6),
        Component("C3", "C", ("n1", "n3"), value=1e-6),
        Component("D3", "D", ("n2", "n3"), model="DSCH"),
        Component("D4", "D", ("n3", "out"), model="DSCH"),
        Component("C4", "C", ("out", "n2"), value=1e-6),
        Component("C5", "C", ("out", "0"), value=0.47e-6),
        Component("D5", "D", ("0", "out"), model="DZ33"),
    ]
    return Circuit(name="witness: 2-stage Cockcroft-Walton multiplier + zener clamp", components=comps)


def spec_witness() -> Circuit:
    """A hand-made circuit meant to meet the active specification (feasibility evidence; never shown to the
    model). For specs without a dedicated one, the P1 witness."""
    return {"P2": _stacked_clamp_witness, "P3": _multiplier_witness}.get(spec.ACTIVE, witness_circuit)()


In [ ]:
%%writefile piezo_synth/sim.py
"""Run LTspice netlists in parallel threads, with a persistent content-hash cache and timing."""
from __future__ import annotations

import hashlib
import json
import threading
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple


@dataclass
class SimResult:
    ok: bool
    meas: dict
    error: str
    seconds: float
    folder: str
    cached: bool = False


def ltspice_backend() -> Callable[[str, str, float], Any]:
    """The project's LTspice helper (in the ltspice-cpu image), imported on first use."""
    import sys
    if "/opt/ltspice" not in sys.path:
        sys.path.append("/opt/ltspice")
    from ltspice_run import simulate  # type: ignore

    def backend(text: str, folder: str, timeout: float):
        return simulate(text, folder, timeout=timeout)
    return backend


class Simulator:
    """backend(netlist_text, folder, timeout) -> object with .ok, .meas, .error (ltspice_run.simulate)."""

    def __init__(self, backend: Callable[[str, str, float], Any], root: Path, workers: int = 8,
                 timeout: float = 120.0, keep_raw: bool = False):
        self.backend = backend
        self.keep_raw = keep_raw     # waveform files are deleted once the measurements are read
        self.root = Path(root)
        self.workers = max(1, int(workers))
        self.timeout = timeout
        self.root.mkdir(parents=True, exist_ok=True)
        self._cache_file = self.root / "sim_cache.jsonl"
        self._cache: Dict[str, SimResult] = {}
        self._lock = threading.Lock()
        self.stats = {"n_sims": 0, "cache_hits": 0, "sim_seconds": 0.0, "wall_seconds": 0.0, "failures": 0}
        if self._cache_file.exists():
            for line in self._cache_file.read_text(encoding="utf-8").splitlines():
                try:
                    d = json.loads(line)
                    self._cache[d["key"]] = SimResult(**d["result"])
                except (ValueError, KeyError, TypeError):
                    continue

    @staticmethod
    def key(text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()

    def _one(self, key: str, text: str) -> SimResult:
        folder = self.root / "sims" / key[:16]
        t = time.time()
        try:
            r = self.backend(text, str(folder), self.timeout)
            res = SimResult(bool(getattr(r, "ok", False)), dict(getattr(r, "meas", {}) or {}),
                            str(getattr(r, "error", "") or ""), round(time.time() - t, 3), str(folder))
            cacheable = True
        except Exception as e:   # a crashed backend must not stop the search
            res = SimResult(False, {}, f"{type(e).__name__}: {e}\n{traceback.format_exc(limit=2)}",
                            round(time.time() - t, 3), str(folder))
            cacheable = False
        if not self.keep_raw:
            self._drop_raw(folder)
        with self._lock:
            self.stats["n_sims"] += 1
            self.stats["sim_seconds"] += res.seconds
            self.stats["failures"] += 0 if res.ok else 1
            if cacheable:
                self._cache[key] = res
            if cacheable and res.ok:   # failures (timeouts, Wine trouble) are retried by a later run here
                with self._cache_file.open("a", encoding="utf-8") as f:
                    f.write(json.dumps({"key": key, "result": asdict(res)}) + "\n")
        return res

    @staticmethod
    def _drop_raw(folder: Path) -> None:
        """The .raw waveforms are the bulk of a simulation folder (MB each, tens of thousands per
        campaign) and are not needed once the .meas results are in hand; netlist and log stay."""
        if not folder.exists():
            return
        for f in folder.rglob("*.raw"):
            try:
                f.unlink()
            except OSError:
                pass

    def run(self, jobs: List[Tuple[str, str]]) -> List[SimResult]:
        """jobs: (label, netlist_text). Results come back in input order."""
        t = time.time()
        keys = [self.key(text) for _, text in jobs]
        todo: Dict[str, str] = {}
        with self._lock:
            for k, (_, text) in zip(keys, jobs):
                if k not in self._cache and k not in todo:
                    todo[k] = text
        fresh: Dict[str, SimResult] = {}
        if todo:
            with ThreadPoolExecutor(max_workers=min(self.workers, len(todo))) as ex:
                futs = {k: ex.submit(self._one, k, text) for k, text in todo.items()}
                for k, f in futs.items():
                    fresh[k] = f.result()
        out = []
        returned = set()
        with self._lock:
            for k in keys:
                if k in fresh and k not in returned:
                    returned.add(k)
                    out.append(fresh[k])
                else:
                    base = fresh.get(k) or self._cache[k]
                    if k not in fresh:
                        self.stats["cache_hits"] += 1
                    out.append(SimResult(base.ok, dict(base.meas), base.error, base.seconds, base.folder,
                                         cached=True))
            self.stats["wall_seconds"] += time.time() - t
        return out


In [ ]:
%%writefile piezo_synth/evaluate.py
"""Turn simulation measurements into pass/fail checks against the spec, and a scalar score."""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from .spec import CRITERIA, Condition, Criterion


@dataclass
class Check:
    condition: str
    criterion: str
    measured: Optional[float]
    limit: float
    op: str
    margin: float
    passed: bool


@dataclass
class Evaluation:
    checks: List[Check] = field(default_factory=list)
    meas: Dict[str, dict] = field(default_factory=dict)
    sim_ok: Dict[str, bool] = field(default_factory=dict)
    sim_error: Dict[str, str] = field(default_factory=dict)
    scalar: float = 0.0
    cond_pass: int = 0
    checks_pass: int = 0
    all_pass: bool = False


def _num(v) -> Optional[float]:
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return f if math.isfinite(f) else None


def _clip(v: float, lo: float, hi: float) -> float:
    return min(max(v, lo), hi)


def evaluate(results: dict, conditions: List[Condition], criteria: List[Criterion] = CRITERIA) -> Evaluation:
    """results: condition id -> SimResult. Missing results count as failed simulations."""
    ev = Evaluation()
    for cond in conditions:
        r = results.get(cond.id)
        ok = bool(r is not None and r.ok)
        ev.sim_ok[cond.id] = ok
        if not ok:
            ev.sim_error[cond.id] = (r.error if r is not None else "not simulated") or "simulation failed"
        meas = {k: _num(v) for k, v in (r.meas.items() if ok else [])}
        if meas.get("vavg") is not None and meas.get("vavg_prev") is not None:
            meas["dv"] = abs(meas["vavg"] - meas["vavg_prev"])
        ev.meas[cond.id] = meas
        all_ok = True
        for crit in criteria:
            m = meas.get(crit.meas)
            if m is None:
                margin = -1.0
            elif crit.op == ">=":
                margin = (m - crit.limit) / crit.scale
            else:
                margin = (crit.limit - m) / crit.scale
            passed = m is not None and margin >= 0
            all_ok = all_ok and passed
            ev.checks.append(Check(cond.id, crit.id, m, crit.limit, crit.op, margin, passed))
        ev.cond_pass += int(all_ok)
    ev.checks_pass = sum(c.passed for c in ev.checks)
    ev.all_pass = bool(ev.checks) and ev.checks_pass == len(ev.checks)
    ev.scalar = (10 * ev.cond_pass + ev.checks_pass
                 + sum(_clip(c.margin, -1, 0) for c in ev.checks)
                 + 0.05 * sum(_clip(c.margin, 0, 0.5) for c in ev.checks))
    return ev


_UNITS = {c.id: c.unit for c in CRITERIA}


def diagnose(ev: Evaluation, limit: int = 8) -> str:
    """Failing checks, worst first, then simulation failures. Empty string if everything passed."""
    failed_sim = {k for k, ok in ev.sim_ok.items() if not ok}
    bad = sorted((c for c in ev.checks if not c.passed and c.condition not in failed_sim),
                 key=lambda c: c.margin)
    lines = []
    for c in bad[:limit]:
        u = _UNITS.get(c.criterion, "")
        got = "not reached / not measured" if c.measured is None else f"{c.measured:.4g} {u}"
        lines.append(f"{c.condition} {c.criterion}: {got} (needs {c.op} {c.limit:g} {u}, margin {c.margin:+.2f})")
    if len(bad) > limit:
        lines.append(f"... and {len(bad) - limit} more failing checks")
    for k in sorted(failed_sim):
        err = " ".join(ev.sim_error.get(k, "").split())[:200]
        lines.append(f"{k}: simulation failed: {err}")
    return "\n".join(lines)


def is_better(a: Evaluation, b: Optional[Evaluation], ncomp_a: int, ncomp_b: int) -> bool:
    """Higher scalar wins; on a tie, fewer components."""
    if b is None:
        return True
    if a.scalar > b.scalar + 1e-9:
        return True
    if abs(a.scalar - b.scalar) <= 1e-9:
        return ncomp_a < ncomp_b
    return False


In [ ]:
%%writefile piezo_synth/bench.py
"""Evaluate circuits on a set of conditions: build the netlists, simulate them in one parallel batch."""
from __future__ import annotations

from typing import List

from .circuit import Circuit
from .evaluate import Evaluation, evaluate
from .netlist import build_netlist


def evaluate_many(circuits: List[Circuit], conditions, sim, labels: List[str]) -> List[Evaluation]:
    """All circuits x conditions go to the simulator as one batch, so they run in parallel."""
    jobs = [(f"{lab}/{cond.id}", build_netlist(c, cond, title=lab))
            for c, lab in zip(circuits, labels) for cond in conditions]
    results = sim.run(jobs) if jobs else []
    n = len(conditions)
    return [evaluate({cond.id: r for cond, r in zip(conditions, results[k * n:(k + 1) * n])}, conditions)
            for k in range(len(circuits))]


def evaluate_circuit(c: Circuit, conditions, sim, label: str) -> Evaluation:
    return evaluate_many([c], conditions, sim, [label])[0]


In [ ]:
%%writefile piezo_synth/screen.py
"""Fast screen of an untuned topology: net power into a storage node held at 1.8 V (see spec.SCREEN).

A few value presets are tried (the proposed values, LC-resonance switch timings, random spreads) on
the screen conditions with short simulations; the best preset is kept as the starting point for
sizing. The screen is a necessary condition for vout_min, relaxed by alpha because values are untuned.
"""
from __future__ import annotations

import math
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from . import spec
from .circuit import Circuit, design_hash, get_param, numeric_params, with_param
from .netlist import build_screen_netlist


def p_fbr_ideal(cond: spec.Condition) -> float:
    """Maximum power of an ideal (lossless-diode) full-bridge rectifier: f*Cp*Voc^2 = Ip^2/(4 pi^2 f Cp)."""
    return cond.ip ** 2 / (4 * math.pi ** 2 * cond.freq * cond.cp)


def p_required(cond: spec.Condition) -> float:
    return spec.SCREEN["vstore"] ** 2 / cond.rl


@dataclass
class ScreenResult:
    passed: bool
    circuit: Circuit                      # best preset
    worst_ratio: float                    # min over screen conditions of p_net / p_required
    ratio: Dict[str, float] = field(default_factory=dict)
    p_net: Dict[str, Optional[float]] = field(default_factory=dict)
    p_req: Dict[str, float] = field(default_factory=dict)
    gain: Dict[str, Optional[float]] = field(default_factory=dict)   # p_net / ideal-FBR maximum
    n_presets: int = 0
    n_sims: int = 0
    sim_failed: bool = False
    error: str = ""

    def summary(self) -> str:
        parts = []
        for cid in sorted(self.ratio):
            pn = self.p_net.get(cid)
            got = "simulation failed" if pn is None else f"{pn * 1e6:.3g} uW"
            parts.append(f"{cid}: {got} of {self.p_req[cid] * 1e6:.3g} uW needed ({self.ratio[cid] * 100:.0f} %)")
        return "; ".join(parts)


def physics_presets(c: Circuit) -> List[Circuit]:
    """Switch on-times from the LC resonance with the nominal piezo capacitance: a full inversion
    (pi*sqrt(L*Cp)) and a quarter period (charge extraction). Empty if there is no SYNC or no inductor."""
    syncs = [x for x in c.components if x.type == "SYNC"]
    inds = sorted((x for x in c.components if x.type == "L"), key=lambda x: x.id)
    if not syncs or not inds:
        return []
    cp = spec.search_conditions()[0].cp
    res = math.pi * math.sqrt(inds[0].value * cp)
    out = []
    for ton in (res, res / 2):
        p = c
        for s in syncs:
            p = with_param(p, s.id, "ton", ton)
        out.append(p)
    return out


def random_preset(c: Circuit, rng: random.Random, spread: float) -> Circuit:
    p = c
    for cid, fld in numeric_params(c):
        v = get_param(p, cid, fld)
        if fld == "phase":
            p = with_param(p, cid, fld, v + rng.uniform(-0.1, 0.1))
        else:
            p = with_param(p, cid, fld, v * math.exp(rng.uniform(-math.log(spread), math.log(spread))))
    return p


def presets(c: Circuit, n: int, rng: random.Random, spread: float) -> List[Circuit]:
    out, seen = [], set()
    for p in [c, *physics_presets(c)]:
        if design_hash(p) not in seen and len(out) < max(1, n):
            seen.add(design_hash(p))
            out.append(p)
    tries = 0
    while len(out) < n and numeric_params(c) and tries < 10 * n:
        tries += 1
        p = random_preset(c, rng, spread)
        if design_hash(p) not in seen:
            seen.add(design_hash(p))
            out.append(p)
    return out


def screen(c: Circuit, sim, rng: random.Random, label: str, conditions=None, n_presets: Optional[int] = None,
           alpha: Optional[float] = None) -> ScreenResult:
    conds = [spec.condition(cid) for cid in (conditions or spec.SCREEN["conditions"])]
    alpha = spec.SCREEN["alpha"] if alpha is None else alpha
    ps = presets(c, n_presets or spec.SCREEN["presets"], rng, spec.SCREEN["spread"])
    jobs = [(f"{label}-screen{k}/{cond.id}", build_screen_netlist(p, cond, title=f"{label}-screen{k}"))
            for k, p in enumerate(ps) for cond in conds]
    results = sim.run(jobs)
    k_res = len(conds)
    best: Optional[ScreenResult] = None
    any_ok = False
    errors = []
    for k, p in enumerate(ps):
        ratio, p_net, p_req, gain = {}, {}, {}, {}
        for j, cond in enumerate(conds):
            r = results[k * k_res + j]
            p_req[cond.id] = p_required(cond)
            isn = r.meas.get("isn") if r.ok else None
            try:
                isn = float(isn)
            except (TypeError, ValueError):
                isn = None
            if isn is None or not math.isfinite(isn):
                errors.append(r.error or "no isn measurement")
                p_net[cond.id], gain[cond.id], ratio[cond.id] = None, None, -math.inf
                continue
            any_ok = True
            pn = spec.SCREEN["vstore"] * isn / spec.SCREEN["rsense"]
            p_net[cond.id] = pn
            gain[cond.id] = pn / p_fbr_ideal(cond)
            ratio[cond.id] = pn / p_req[cond.id]
        worst = min(ratio.values())
        cand = ScreenResult(worst >= alpha, p, worst, ratio, p_net, p_req, gain)
        if best is None or worst > best.worst_ratio:
            best = cand
    best.n_presets, best.n_sims = len(ps), len(jobs)
    best.sim_failed = not any_ok
    if best.sim_failed:
        best.error = " ".join((errors[0] if errors else "simulation failed").split())[:200]
    return best


In [ ]:
%%writefile piezo_synth/sizer.py
"""Value sizing for a fixed topology: an elitist (1+lambda) evolution strategy in log-parameter space.

Each generation is one batch of lambda designs, simulated in parallel. The step size follows a
success rule (grow after an improving generation, shrink otherwise). The sizer is resumable: the
search runs generations while the model is thinking and continues the same sizer on the next wait.
Like simulation-in-loop feasibility sizing, it stops as soon as a design meets the whole spec.
"""
from __future__ import annotations

import math
import random
from typing import List, Optional

from .circuit import Circuit, design_hash, get_param, numeric_params, with_param
from .evaluate import Evaluation, is_better

SIGMA0 = 0.6          # log-space step: one sigma is a factor of ~1.8
SIGMA_MIN = 0.02
GROW, SHRINK = 1.3, 0.82
PHASE_SCALE = 0.1     # phase is additive: one sigma moves it by PHASE_SCALE * sigma periods


class Sizer:
    def __init__(self, circuit: Circuit, start_ev: Evaluation, budget: int, rng: random.Random, lam: int = 8,
                 sigma0: float = SIGMA0):
        self.params = numeric_params(circuit)
        self.best, self.best_ev = circuit, start_ev
        self.ncomp = len(circuit.components)
        self.budget, self.rng, self.lam = budget, rng, max(1, lam)
        self.sigma = sigma0
        self.n_evals = self.generations = 0
        self.seen = {design_hash(circuit)}
        self.done, self.reason = False, ""
        self._check_done()

    def _check_done(self) -> None:
        if self.done:
            return
        if self.best_ev.all_pass:
            self.done, self.reason = True, "spec_met"
        elif not self.params:
            self.done, self.reason = True, "no_parameters"
        elif self.n_evals >= self.budget:
            self.done, self.reason = True, "budget"
        elif self.sigma < SIGMA_MIN:
            self.done, self.reason = True, "converged"

    def _mutate(self) -> Circuit:
        c = self.best
        for cid, fld in self.params:
            v = get_param(c, cid, fld)
            z = self.rng.gauss(0.0, 1.0) * self.sigma
            c = with_param(c, cid, fld, v + PHASE_SCALE * z if fld == "phase" else v * math.exp(z))
        return c

    def ask(self) -> List[Circuit]:
        """The next generation (distinct, never-seen designs). Empty when done."""
        if self.done:
            return []
        n = min(self.lam, self.budget - self.n_evals)
        out, tries = [], 0
        while len(out) < n and tries < 50 * n:
            tries += 1
            c = self._mutate()
            h = design_hash(c)
            if h not in self.seen:
                self.seen.add(h)
                out.append(c)
        if not out:                      # every mutation clamps onto known designs: nothing left to try
            self.done, self.reason = True, "converged"
        return out

    def tell(self, circuits: List[Circuit], evs: List[Evaluation]) -> Optional[Circuit]:
        """Record a generation's evaluations; returns the new best circuit if it improved."""
        self.n_evals += len(circuits)
        self.generations += 1
        improved = None
        for c, ev in zip(circuits, evs):
            if is_better(ev, self.best_ev, self.ncomp, self.ncomp):
                self.best, self.best_ev, improved = c, ev, c
        self.sigma *= GROW if improved is not None else SHRINK
        self._check_done()
        return improved


In [ ]:
%%writefile piezo_synth/diagnostics.py
"""Physical interpretation of results for the model: turns measurements and screen powers into short
statements about what is wrong (no output, too little power, overvoltage, slow start, ...).

The statements name the symptom and the part of the circuit's job that fails, not a specific fix, so
the feedback does not steer the model toward a known topology.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional

from . import spec
from .evaluate import Evaluation



def dead_v() -> float:
    """Below this peak output on every condition the circuit is considered pinned (no charge arrives)."""
    return 0.5 * spec.vmin()

MAJOR = ("pinned", "sim_failed", "underpower", "overvoltage")   # codes tracked for repeated failures


@dataclass(frozen=True)
class Hint:
    code: str
    text: str


def _label(cid: str) -> str:
    try:
        return f"{cid} ({spec.condition(cid).label})"
    except KeyError:
        return cid


def _list(items: Iterable[str], n: int = 3) -> str:
    items = list(items)
    return ", ".join(items[:n]) + (f" and {len(items) - n} more" if len(items) > n else "")


def hints(ev: Optional[Evaluation] = None, screen=None) -> List[Hint]:
    out: List[Hint] = []
    VMIN, VMAX = spec.vmin(), spec.vmax()
    DEAD_V = dead_v()
    _LIM = {c.id: c.limit for c in spec.CRITERIA}
    if ev is not None:
        failed = [k for k, ok in ev.sim_ok.items() if not ok]
        if failed:
            err = " ".join(ev.sim_error.get(failed[0], "").split())[:160]
            out.append(Hint("sim_failed", f"the simulation failed on {_list(failed)} ({err}); the circuit may "
                                          f"have a short, a loop of ideal sources/switches, or a floating section"))
        ok = {k: m for k, m in ev.meas.items() if ev.sim_ok.get(k)}
        vmax = {k: m.get("vmax") for k, m in ok.items() if m.get("vmax") is not None}
        if ok and vmax and max(vmax.values()) < DEAD_V:
            out.append(Hint("pinned", f"V(out) never exceeds {max(vmax.values()):.2g} V on any condition: no "
                                      f"charge reaches the storage, or something conducts from out to ground "
                                      f"(check every diode's orientation: nodes are [anode, cathode])"))
        else:
            low = [(k, m["vavg"]) for k, m in ok.items() if m.get("vavg") is not None and m["vavg"] < VMIN
                   and (m.get("vmax") or 0) <= VMAX]
            if low:
                out.append(Hint("underpower", f"the output settles below {VMIN:g} V on " +
                                _list(f"{_label(k)} at {v:.2f} V" for k, v in low) +
                                ": the interface does not deliver enough power there"))
            high = [(k, v) for k, v in vmax.items() if v > VMAX]
            if high:
                out.append(Hint("overvoltage", f"V(out) exceeds {VMAX:g} V on " +
                                _list(f"{_label(k)} (peak {v:.2f} V)" for k, v in high) +
                                ": the output needs limiting when the source is strong or the load light"))
            slow = [(k, m.get("tup")) for k, m in ok.items() if (m.get("vavg") or 0) >= VMIN
                    and (m.get("tup") is None or m["tup"] > _LIM["startup"])]
            if slow:
                out.append(Hint("slow_start", "start-up is too slow on " +
                                _list(f"{k} ({'never' if t is None else f'{t:.2f} s'})" for k, t in slow) +
                                ": the storage charges too slowly from 0 V (too much capacitance or too "
                                "little power at low voltage)"))
            drift = [(k, m.get("dv")) for k, m in ok.items() if (m.get("vavg") or 0) >= VMIN
                     and m.get("dv") is not None and m["dv"] > _LIM["settled"]]
            if drift:
                out.append(Hint("not_settled", "the output is still drifting at 0.8 s on " +
                                _list(f"{k} ({d * 1e3:.0f} mV between windows)" for k, d in drift) +
                                ": the storage time constant is too long"))
            rip = [(k, m.get("vpp")) for k, m in ok.items() if m.get("vpp") is not None and m["vpp"] > _LIM["ripple"]]
            if rip:
                out.append(Hint("ripple", "ripple is too large on " +
                                _list(f"{k} ({v * 1e3:.0f} mV)" for k, v in rip) +
                                ": the storage cannot buffer the charge pulses against the load"))
    if screen is not None:
        if screen.sim_failed:
            out.append(Hint("sim_failed", f"the screening simulation failed ({screen.error})"))
        else:
            short = [(k, r) for k, r in screen.ratio.items() if r < 1.0]
            if short and not any(h.code == "underpower" for h in out):
                out.append(Hint("underpower", f"with the output fixed at {spec.SCREEN['vstore']:g} V, the net power delivered covers only " +
                                _list(f"{r * 100:.0f} % of the load's need on {_label(k)}" for k, r in short) +
                                " (best of the value presets tried)"))
    return out


def codes(hs: List[Hint]) -> set:
    return {h.code for h in hs}


def repeated(history: List[set], n: int = 3) -> List[str]:
    """Major problem codes shared by the last n results (empty if fewer than n)."""
    if len(history) < n:
        return []
    common = set.intersection(*[set(h) for h in history[-n:]])
    return [c for c in MAJOR if c in common]


def hint_text(hs: List[Hint]) -> str:
    return "\n".join(f"- {h.text}" for h in hs)


In [ ]:
%%writefile piezo_synth/robust.py
"""Monte Carlo robustness of a design under component tolerances and controller timing error."""
from __future__ import annotations

import random
from typing import List

from . import spec
from .bench import evaluate_many
from .circuit import Circuit, get_param, numeric_params, with_param


def perturb(c: Circuit, rng: random.Random, tol: dict = None) -> Circuit:
    tol = spec.TOLERANCES if tol is None else tol
    types = {x.id: x.type for x in c.components}
    p = c
    for cid, fld in numeric_params(c):
        v = get_param(p, cid, fld)
        if fld == "phase":
            p = with_param(p, cid, fld, v + rng.uniform(-tol["phase"], tol["phase"]))
        else:
            t = tol["ton"] if fld == "ton" else tol.get(types[cid], 0.0)
            p = with_param(p, cid, fld, v * (1 + rng.uniform(-t, t)))
    return p


def monte_carlo(c: Circuit, conditions, sim, rng: random.Random, n: int, label: str) -> dict:
    samples: List[Circuit] = [perturb(c, rng) for _ in range(n)]
    evs = evaluate_many(samples, conditions, sim, [f"{label}-mc{k:02d}" for k in range(n)])
    per = {cond.id: 0 for cond in conditions}
    for ev in evs:
        for cond in conditions:
            if all(ch.passed for ch in ev.checks if ch.condition == cond.id):
                per[cond.id] += 1
    return {"n": n, "pass_all": sum(ev.all_pass for ev in evs) / n if n else 0.0,
            "per_condition": {k: v / n for k, v in per.items()} if n else {},
            "tolerances": dict(spec.TOLERANCES),
            "worst_condition": min(per, key=per.get) if per else None}


In [ ]:
%%writefile piezo_synth/llm.py
"""Client for the cluster's Gemma 4 31B (vLLM, OpenAI-compatible API), and a scripted stand-in."""
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from typing import Callable, List, Optional

URL = "http://gemma-4-31b.cpd-instance.svc:3000/v1/chat/completions"
MODEL = "google/gemma-4-31b"


@dataclass
class LLMReply:
    answer: str = ""
    reasoning: str = ""
    seconds: float = 0.0
    prompt_tokens: Optional[int] = None
    completion_tokens: Optional[int] = None
    reasoning_tokens: Optional[int] = None
    finish: Optional[str] = None
    error: Optional[str] = None


def _requests_post(url, json, timeout):   # imported lazily: only needed for real calls
    import requests
    return requests.post(url, json=json, timeout=timeout)


class _HTTPError(Exception):
    def __init__(self, status: int, text: str):
        super().__init__(f"HTTP {status}: {text[:300]}")
        self.status = status


class GemmaClient:
    def __init__(self, url: str = URL, model: str = MODEL, timeout: float = 900, retries: int = 3,
                 backoff: float = 10.0, post: Callable = _requests_post, sleep: Callable = time.sleep):
        self.url, self.model, self.timeout = url, model, timeout
        self.retries, self.backoff = max(1, retries), backoff
        self.post, self.sleep = post, sleep

    def ask(self, messages: List[dict], think: bool, max_tokens: Optional[int] = None,
            temperature: Optional[float] = None, timeout: Optional[float] = None) -> LLMReply:
        """Never raises: on repeated failure the reply carries `error`. `timeout` (s) caps this call's
        read timeout, e.g. to the remaining wall-clock budget. A read timeout is not retried: resending
        the same request would take as long again."""
        read_timeout = self.timeout if timeout is None else max(1.0, min(self.timeout, timeout))
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens or (16384 if think else 2048),
            "temperature": temperature if temperature is not None else (1.0 if think else 0.3),
            "chat_template_kwargs": {"enable_thinking": bool(think)},
        }
        t = time.time()
        last = ""
        for attempt in range(1, self.retries + 1):
            try:
                r = self.post(self.url, json=body, timeout=read_timeout)
                status = getattr(r, "status_code", 200)
                if status >= 400:
                    text = getattr(r, "text", "") or ""
                    raise _HTTPError(status, text if isinstance(text, str) else "")
                out = r.json()
                choice = out["choices"][0]
                msg = choice.get("message") or {}
                usage = out.get("usage") or {}
                details = usage.get("completion_tokens_details") or {}
                return LLMReply(
                    answer=(msg.get("content") or "").strip(),
                    reasoning=msg.get("reasoning") or msg.get("reasoning_content") or "",
                    seconds=round(time.time() - t, 1),
                    prompt_tokens=usage.get("prompt_tokens"),
                    completion_tokens=usage.get("completion_tokens"),
                    reasoning_tokens=details.get("reasoning_tokens"),
                    finish=choice.get("finish_reason"),
                )
            except _HTTPError as e:
                last = str(e)
                if e.status < 500 and e.status != 429:   # 429 = rate limited: worth waiting and retrying
                    break
            except Exception as e:   # connection errors, timeouts, malformed bodies
                last = f"{type(e).__name__}: {e}"
                if "Timeout" in type(e).__name__:
                    break
            if attempt < self.retries:
                self.sleep(self.backoff * attempt)
        return LLMReply(seconds=round(time.time() - t, 1), error=last or "unknown error")


class ScriptedLLM:
    """Returns pre-written replies in order (repeating the last). For tests and --fake dry runs."""

    def __init__(self, replies: list):
        self.replies = list(replies)
        self.received: List[List[dict]] = []
        self._i = 0

    def ask(self, messages: List[dict], think: bool, max_tokens: Optional[int] = None,
            temperature: Optional[float] = None, timeout: Optional[float] = None) -> LLMReply:
        self.received.append(list(messages))
        r = self.replies[min(self._i, len(self.replies) - 1)]
        self._i += 1
        if isinstance(r, LLMReply):
            return r
        return LLMReply(answer=str(r), seconds=0.0, prompt_tokens=100, completion_tokens=50,
                        reasoning_tokens=None, finish="stop")


def json_from(text: str) -> Optional[dict]:
    """The circuit JSON in a reply: a fenced ```json block first, else the first object containing
    "components", else the first JSON object. None if there is none."""
    if not text:
        return None
    for m in re.finditer(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S):
        try:
            d = json.loads(m.group(1))
            if isinstance(d, dict):
                return d
        except ValueError:
            pass
    dec = json.JSONDecoder()
    first = None
    for m in re.finditer(r"\{", text):
        try:
            d, _end = dec.raw_decode(text, m.start())
        except ValueError:
            continue
        if isinstance(d, dict):
            if "components" in d:
                return d
            if first is None:
                first = d
    return first


def _top_level_json(text: str) -> list:
    """Every top-level JSON value (object or array) in the text: fenced blocks first, then bare ones."""
    out = []
    for m in re.finditer(r"```(?:json)?\s*([\[{].*?[\]}])\s*```", text, re.S):
        try:
            out.append(json.loads(m.group(1)))
        except ValueError:
            pass
    dec = json.JSONDecoder()
    i = 0
    while i < len(text):
        m = re.compile(r"[\[{]").search(text, i)
        if not m:
            break
        try:
            d, end = dec.raw_decode(text, m.start())
        except ValueError:
            i = m.start() + 1
            continue
        out.append(d)
        i = end
    return out


def proposals_from(text: str) -> Optional[list]:
    """The proposed circuits in a reply: {"proposals": [...]}, a bare circuit object, or a list of them.
    None if the reply holds no such JSON."""
    if not text:
        return None
    values = _top_level_json(text)
    for v in values:
        if isinstance(v, dict) and isinstance(v.get("proposals"), list):
            return [x for x in v["proposals"] if isinstance(x, dict)]
    for v in values:
        if isinstance(v, list) and any(isinstance(x, dict) and "components" in x for x in v):
            return [x for x in v if isinstance(x, dict)]
    for v in values:
        if isinstance(v, dict) and "components" in v:
            return [v]
    return None


In [ ]:
%%writefile piezo_synth/baselines.py
"""Model-free baseline: proposes random valid circuits over the same vocabulary, through the same
screen -> evaluation -> sizing pipeline, so a run with it measures what the language model adds."""
from __future__ import annotations

import json
import math
import random
from typing import List, Optional

from .circuit import DIODE_MODELS, EDGES, Circuit, Component, to_dict, validate
from .llm import LLMReply
from .spec import LIMITS

TYPE_WEIGHTS = {"D": 0.35, "C": 0.2, "L": 0.15, "S": 0.15, "R": 0.15}


def _logu(rng: random.Random, lo: float, hi: float) -> float:
    return math.exp(rng.uniform(math.log(lo), math.log(hi)))


def _attempt(rng: random.Random) -> Circuit:
    internal = [f"n{k}" for k in range(1, rng.randint(0, 3) + 1)]
    pool = ["0", "pzp", "pzn", "out", *internal]
    n = rng.randint(4, 12)
    types = rng.choices(list(TYPE_WEIGHTS), weights=list(TYPE_WEIGHTS.values()), k=n)
    n_sw = 0
    comps: List[Component] = []
    count = {}
    for t in types:
        if t == "S" and n_sw >= LIMITS["max_switches"]:
            t = "D"
        a, b = rng.sample(pool, 2)
        count[t] = count.get(t, 0) + 1
        cid = f"{t}{count[t]}"
        if t in ("R", "C", "L"):
            comps.append(Component(cid, t, (a, b), value=_logu(rng, *LIMITS[t])))
        elif t == "D":
            comps.append(Component(cid, t, (a, b), model=rng.choice(DIODE_MODELS)))
        else:
            n_sw += 1
            comps.append(Component(cid, t, (a, b), ctrl="G1"))
    if n_sw:
        comps.append(Component("G1", "SYNC", (), ton=_logu(rng, *LIMITS["ton"]),
                               phase=rng.uniform(*LIMITS["phase"]), edges=rng.choice(EDGES)))
    return Circuit(name="random", components=comps, move="new_topology", rationale="random baseline")


def random_circuit(rng: random.Random, max_tries: int = 20000) -> Circuit:
    for _ in range(max_tries):
        c = _attempt(rng)
        if not validate(c):
            return c
    raise RuntimeError("no valid random circuit found")


class RandomProposer:
    """Stands in for the model: same ask() interface, zero tokens, instant replies."""

    def __init__(self, seed: int = 1, per_round: int = 3):
        self.rng = random.Random(seed)
        self.per_round = per_round
        self.n = 0

    def ask(self, messages, think: bool, max_tokens: Optional[int] = None, temperature: Optional[float] = None,
            timeout: Optional[float] = None) -> LLMReply:
        props = []
        for _ in range(self.per_round):
            self.n += 1
            c = random_circuit(self.rng)
            c.name = f"random-{self.n:03d}"
            props.append(to_dict(c))
        return LLMReply(answer=json.dumps({"proposals": props}), seconds=0.0, prompt_tokens=0,
                        completion_tokens=0, reasoning_tokens=0, finish="stop")


In [ ]:
%%writefile piezo_synth/prompts.py
"""Prompts for the model. Rebuilt from the search state every iteration (no growing chat history).

Only search conditions appear here: the held-out conditions are never shown to the model.
"""
from __future__ import annotations

import json
import math
from typing import List, Optional

from . import spec
from .circuit import DIODE_MODELS, MOVES
from .evaluate import Evaluation

EXAMPLE_CIRCUIT = {
    "move": "new_topology",
    "name": "short label for this design",
    "rationale": "what you changed and why it should fix the failing checks",
    "done": False,
    "components": [
        {"id": "D1", "type": "D", "nodes": ["pzp", "out"], "model": "DSCH"},
        {"id": "C1", "type": "C", "nodes": ["out", "0"], "value": 2.2e-6},
        {"id": "L1", "type": "L", "nodes": ["pzp", "n1"], "value": 4.7e-3},
        {"id": "S1", "type": "S", "nodes": ["n1", "pzn"], "ctrl": "G1"},
        {"id": "G1", "type": "SYNC", "ton": 3.4e-5, "phase": 0.0, "edges": "both"},
    ],
}
EXAMPLE_JSON = {"proposals": [EXAMPLE_CIRCUIT]}

# The default (neutral) example: the same JSON syntax with a meaningless R-C network, so the example
# carries no piece of a known interface. EXAMPLE_CIRCUIT (hinted variant only) is the SSHI branch.
NEUTRAL_EXAMPLE_JSON = {"proposals": [{
    "move": "new_topology",
    "name": "short label for this design",
    "rationale": "what you changed and why it should fix the failing checks",
    "done": False,
    "components": [
        {"id": "R1", "type": "R", "nodes": ["pzp", "n1"], "value": 1e3},
        {"id": "C1", "type": "C", "nodes": ["n1", "out"], "value": 1e-6},
        {"id": "C2", "type": "C", "nodes": ["out", "pzn"], "value": 1e-6},
    ],
}]}

_DIODE_WIRING = """     Orientation matters:
       full bridge: ["pzp", "out"], ["pzn", "out"], ["0", "pzp"], ["0", "pzn"] (return diodes have
       their anode on ground);
       zener clamp on out: ["0", "out"] (cathode on out, so it is reverse-biased and breaks down at
       3.3 V). ["out", "0"] would be forward-biased and hold out near 0.7 V; it is rejected."""
_DIODE_WIRING_NEUTRAL = """     Orientation matters: a diode with nodes ["a", "b"] conducts forward from a to b; a DZ33 with
       nodes ["a", "b"] also conducts in reverse once V(b) - V(a) exceeds 3.3 V. A diode forward-biased
       from out to ground would hold out near 0.7 V; it is rejected."""

REPEATED_TEXT = {
    "pinned": "the output stays pinned near 0 V (a wiring or orientation error, not a value problem)",
    "sim_failed": "the simulation fails",
    "underpower": "too little power reaches the output on some conditions",
    "overvoltage": "the output exceeds the maximum voltage on some conditions",
}


def _f(v, fmt="{:.3g}") -> str:
    return "-" if v is None else fmt.format(v)


def condition_table(ev: Evaluation, conditions) -> str:
    """Fixed-width table: measurements and pass/fail per criterion for each condition."""
    crit = [c.id for c in spec.CRITERIA]
    head = f"{'cond':<5}{'vavg V':>8}{'vmax V':>8}{'vpp mV':>8}{'tup s':>8}{'Pload uW':>10}  " + \
        " ".join(f"{c:<8}" for c in crit)
    rows = [head]
    by = {(c.condition, c.criterion): c for c in ev.checks}
    for cond in conditions:
        m = ev.meas.get(cond.id, {})
        if not ev.sim_ok.get(cond.id, False):
            rows.append(f"{cond.id:<5}  simulation failed")
            continue
        vpp = m.get("vpp")
        pl = m.get("pload")
        flags = " ".join(f"{('ok' if by[(cond.id, c)].passed else 'FAIL'):<8}" for c in crit if (cond.id, c) in by)
        rows.append(f"{cond.id:<5}{_f(m.get('vavg')):>8}{_f(m.get('vmax')):>8}"
                    f"{_f(None if vpp is None else vpp * 1e3):>8}{_f(m.get('tup')):>8}"
                    f"{_f(None if pl is None else pl * 1e6):>10}  {flags}")
    return "\n".join(rows)


def system_prompt(reference_eval: Evaluation, per_round: int = 3, screen: bool = True, size_budget: int = 48,
                  hinted: bool = False) -> str:
    """By default the prompt carries no fragment of the textbook answer. hinted=True (the hinted variant)
    restores the original prompt: the SSHI branch as JSON example, its switch-timing formula, and the
    full-bridge and zener-clamp wiring examples."""
    conds = spec.search_conditions()
    sc = spec.SCREEN
    screen_text = (f"""- Each proposal is first SCREENED: with out fixed at {sc['vstore']:g} V, the net power it delivers is
  measured on {', '.join(sc['conditions'])} for its proposed values and a few automatic value presets.
  If the best preset delivers less than {sc['alpha'] * 100:.0f} % of the load's need (V^2/RL) on any of them,
  the design is reported back with those powers and is not tuned or simulated further.
""" if screen else "")
    how_many = ("exactly 1 design" if per_round == 1 else
                f"up to {per_round} designs with DIFFERENT connectivity (different structures, not value variants)")
    ctable = "\n".join(
        f"  {c.id}: {c.label}; Ip = {c.ip * 1e6:g} uA peak, f = {c.freq:g} Hz, Cp = {c.cp * 1e9:g} nF, "
        f"RL = {c.rl / 1e3:g} kOhm, temperature {c.temp:g} C" for c in conds)
    crit = "\n".join(f"  {c.id}: {c.meas} {c.op} {c.limit:g} {c.unit} ({c.description})" for c in spec.CRITERIA)
    L = spec.LIMITS
    nom = conds[0]
    voc_nom = nom.ip / (2 * math.pi * nom.freq * nom.cp)
    ton_hint = "" if not hinted else "\n     For L-Cp inversion, ton is about pi*sqrt(L*Cp)."
    diode_wiring = _DIODE_WIRING if hinted else _DIODE_WIRING_NEUTRAL
    example = json.dumps(EXAMPLE_JSON if hinted else NEUTRAL_EXAMPLE_JSON, indent=1)
    return f"""You are an expert power-electronics designer synthesising the interface circuit of a
piezoelectric energy harvester. You work by experiment: you propose a circuit, it is simulated in
LTspice under several operating conditions, and you receive the measurements.

GOAL
A piezo transducer must power a sensor-node load RL from a storage capacitor at node "out".
Under EVERY operating condition below, starting from 0 V (cold start) and simulating 0.8 s:
{crit}
Measurement windows: vavg and vpp over 0.7-0.8 s; vmax over the whole run; tup = first time
V(out) reaches {spec.vmin():g} V (missing if never); dv = |mean(0.7-0.8 s) - mean(0.6-0.7 s)|.

PIEZO MODEL AND OPERATING CONDITIONS
The piezo is a sinusoidal current source Ip*sin(2*pi*f*t) in parallel with its capacitance Cp and a
20 MOhm leakage resistance, between nodes pzp and pzn (Cp = {nom.cp * 1e9:g} nF at {nom.id}).
Its open-circuit voltage amplitude is Ip/(2*pi*f*Cp), about {voc_nom:.2g} V at {nom.id}; compare it with the
output window and the diode drops.
{ctable}

TEST BENCH (fixed, added automatically - do not include these)
- the piezo model between pzp and pzn; the load RL from out to 0;
- for every SYNC controller you use, a 0.5 uA supply current drawn from out (its power cost);
- a 1 GOhm resistor to 0 from pzp, pzn and every internal node (prevents floating nodes).

COMPONENTS YOU MAY USE (at most {L['max_components']} components)
- R  resistor, value {L['R'][0]:g}..{L['R'][1]:g} Ohm, nodes [a, b]
- C  capacitor, value {L['C'][0]:g}..{L['C'][1]:g} F, nodes [a, b]
- L  inductor, value {L['L'][0]:g}..{L['L'][1]:g} H; it has series resistance 2 Ohm + 5 Ohm per mH
- D  diode, nodes [anode, cathode]: forward current flows from the first node to the second.
     Model one of {', '.join(DIODE_MODELS)}:
     DSCH = low-drop Schottky (~0.25 V at uA), DSI = silicon small-signal (~0.5 V at uA),
     DZ33 = 3.3 V zener (reverse breakdown 3.3 V, conducts ~1 uA at 3.3 V and much more above)
{diode_wiring}
- S  switch, nodes [a, b], "ctrl": id of a SYNC; Ron = 2 Ohm, Roff = 1 GOhm (at most {L['max_switches']})
- SYNC ideal synchronous controller (no nodes, at most {L['max_sync']}): closes its switches for "ton"
     seconds ({L['ton'][0]:g}..{L['ton'][1]:g}) at the piezo voltage extrema, which occur at the zero
     crossings of the piezo current. "edges": "both" (every half period), "pos" (positive peak only)
     or "neg" (negative peak only). "phase" shifts the pulses by that fraction of the excitation
     period ({L['phase'][0]}..{L['phase'][1]}). It tracks the excitation frequency automatically.{ton_hint}
Nodes: "0" is ground; pzp, pzn are the piezo terminals; out is the load/storage node; name any other
nodes yourself (lower-case, e.g. n1, mid). Every node needs at least two connections, and out must
connect to the piezo through your circuit.

REFERENCE DESIGN: FBR (full-bridge rectifier, 4 x DSCH, 2.2 uF on out). Its results:
{condition_table(reference_eval, conds)}

HOW TO WORK
- Component values are optimised automatically (an evolution strategy, up to {size_budget} full
  evaluations per structure, running while you think), so spend your reasoning on circuit
  CONNECTIVITY (topology): which structure can meet every condition. A structure that was already
  explored is not simulated again, whatever its values.
{screen_text}- Each round, propose {how_many}.
- Classify each design with "move": new_topology or modify_topology, and explain the expected effect
  in "rationale". Set "done": true only when you believe no further structural change can improve on
  the best design so far.
- Reply with ONE JSON object only, in this form:
{example}
"""


def required_move(moves: List[str]) -> Optional[str]:
    """Force a structural change after three value-only proposals in a row."""
    if len(moves) >= 3 and all(m == "resize" for m in moves[-3:]):
        return "new_topology or modify_topology"
    return None


def _hints(d: dict) -> str:
    hs = d.get("hints") or []
    return ("\nwhat this means:\n" + "\n".join(f"- {h}" for h in hs)) if hs else ""


def _design_block(d: dict) -> str:
    extra = ""
    if d.get("screen"):
        extra += f"\nscreen (out fixed at {spec.SCREEN['vstore']:g} V): {d['screen']}"
    if d.get("sizing"):
        extra += f"\nvalue sizing: {d['sizing']}"
    return (f"[{d['id']}] {d['name']}  (topology {d['topo_hash']}, score {d['scalar']:.2f}, "
            f"conditions passed {d['cond_pass']}/{len(spec.search_conditions())})\n"
            f"{d['circuit_text']}\n{d['table']}\n"
            f"failing: {d['diagnosis'] or 'nothing - all checks pass'}{extra}{_hints(d)}")


def _result_block(k: int, d: dict) -> str:
    name = d.get("name", "")
    kind = d.get("kind")
    if kind is None:   # older dict form
        kind = ("invalid" if d.get("invalid_problems") else "duplicate" if d.get("duplicate")
                else "error" if d.get("error") else "evaluated")
    head = f"Design {k} '{name}': "
    if kind == "invalid":
        return head + "rejected before simulation:\n- " + "\n- ".join(d["invalid_problems"])
    if kind == "duplicate":
        return head + (f"duplicate: same structure as an earlier design ({d.get('id', '?')}); nothing new was simulated. "
                       f"Values are tuned automatically, so propose a different structure.")
    if kind == "error":
        return f"Your previous request failed: {d['error']}"
    if kind == "screen_failed":
        return (head + f"failed the screen and was not tuned (topology {d.get('topo_hash', '?')}).\n"
                f"{d.get('circuit_text', '')}\nscreen (out fixed at {spec.SCREEN['vstore']:g} V): {d.get('screen', '')}{_hints(d)}")
    return head + "simulated with the best screening preset (it is being tuned):\n" + _design_block(d)


def iteration_prompt(iteration: int, max_rounds: int, top: List[dict], last, explored: List[dict],
                     required: Optional[str], repeated: Optional[List[str]] = None) -> str:
    parts = [f"Round {iteration} of {max_rounds}."]
    if isinstance(last, dict):
        last = [last]
    if not last and iteration <= 1:
        parts.append("This is your first round: choose the structures you think most likely to meet "
                     "every condition.")
    elif last:
        parts.append("RESULTS OF YOUR PREVIOUS ROUND:\n\n" +
                     "\n\n".join(_result_block(k, d) for k, d in enumerate(last, 1)))
    if repeated:
        parts.append("REPEATED PROBLEM: your last three designs all show the same failure: " +
                     "; ".join(REPEATED_TEXT.get(c, c) + f" [{c}]" for c in repeated) +
                     ". Change the part of the circuit responsible for it rather than its values.")
    if top:
        parts.append("BEST DESIGNS SO FAR (values already tuned):\n\n" + "\n\n".join(_design_block(d) for d in top))
    if explored:
        parts.append("Topologies explored so far (hash: name, best score):\n" + "\n".join(
            f"  {e['topo_hash']}: {e['name']} ({e['best_scalar']:.2f})" for e in explored))
    if required:
        parts.append(f"Your last three designs only changed values. This round MUST change the "
                     f"connectivity (move = {required}).")
    parts.append("Reply with one JSON object only.")
    return "\n\n".join(parts)


def repair_messages(system: str, user: str, bad_answer: str, problems: List[str]) -> List[dict]:
    return [{"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": bad_answer[:4000]},
            {"role": "user", "content": "That reply could not be used:\n- " + "\n- ".join(problems) +
             "\nReply again with one corrected JSON object only."}]


In [ ]:
%%writefile piezo_synth/refine.py
"""Value tuning for a fixed topology: coordinate search, multiplicative for R/C/L/ton, additive for phase."""
from __future__ import annotations

import random
from typing import Callable, List, Tuple

from .circuit import Circuit, get_param, numeric_params, with_param
from .evaluate import Evaluation, is_better

MIN_FACTOR = 1.05
MIN_ADD = 0.005


def _initial_step(fld: str) -> float:
    if fld == "phase":
        return 0.05
    if fld == "ton":
        return 1.3
    return 2.0


def refine(circuit: Circuit, start: Evaluation, evaluate_fn: Callable[[Circuit], Evaluation], budget: int,
           rng: random.Random, should_stop: Callable[[], bool] = lambda: False
           ) -> Tuple[Circuit, Evaluation, List[Tuple[Circuit, Evaluation]]]:
    """Return (best circuit, its evaluation, every (circuit, evaluation) tried)."""
    params = numeric_params(circuit)
    tried: List[Tuple[Circuit, Evaluation]] = []
    if not params or budget <= 0:
        return circuit, start, tried
    ncomp = len(circuit.components)
    step = {p: _initial_step(p[1]) for p in params}
    best, best_ev = circuit, start
    seen = set()

    def small(p) -> bool:
        return step[p] < (MIN_ADD if p[1] == "phase" else MIN_FACTOR)

    while len(tried) < budget and not all(small(p) for p in params):
        improved = False
        order = list(params)
        rng.shuffle(order)
        for p in order:
            if small(p):
                continue
            cid, fld = p
            cur = get_param(best, cid, fld)
            if fld == "phase":
                cands = [cur + step[p], cur - step[p]]
            else:
                cands = [cur * step[p], cur / step[p]]
            for v in cands:
                if len(tried) >= budget or should_stop():
                    return best, best_ev, tried
                cand = with_param(best, cid, fld, v)
                newv = get_param(cand, cid, fld)
                key = (p, round(newv, 15), id(best))
                if abs(newv - cur) <= 1e-15 * max(1.0, abs(cur)) or key in seen:
                    continue   # clamped at a bound, or already tried from this point
                seen.add(key)
                ev = evaluate_fn(cand)
                tried.append((cand, ev))
                if is_better(ev, best_ev, ncomp, ncomp):
                    best, best_ev = cand, ev
                    improved = True
                    break
        if not improved:
            for p in params:
                step[p] = step[p] / 2 if p[1] == "phase" else step[p] ** 0.5
    return best, best_ev, tried


In [ ]:
%%writefile piezo_synth/records.py
"""Incremental run record: events, candidates, measurements, checks, progress, timing, model exchanges.

Every write is an append or an atomic replace, so a job that is killed leaves a usable record.
"""
from __future__ import annotations

import csv
import json
import os
import threading
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

from .circuit import Circuit, to_dict
from .evaluate import Evaluation
from .netlist import MEAS_NAMES

CANDIDATE_FIELDS = ["id", "iteration", "origin", "parent", "move", "name", "topo_hash", "design_hash",
                    "n_components", "split", "scalar", "cond_pass", "checks_pass", "all_pass"]
MEAS_FIELDS = ["candidate", "split", "condition", *MEAS_NAMES, "dv", "sim_ok", "sim_error"]
PROPOSAL_FIELDS = ["round", "k", "name", "move", "topo_hash", "status", "candidate", "of", "worst_ratio",
                   *[f"ratio_{c}" for c in ("S2", "S4", "S6")], *[f"gain_{c}" for c in ("S2", "S4", "S6")],
                   "n_presets", "hints", "problems"]
CHECK_FIELDS = ["candidate", "split", "condition", "criterion", "measured", "op", "limit", "margin", "passed"]


@dataclass
class Candidate:
    id: str
    iteration: int
    origin: str            # reference | witness | llm | size | refine | audit
    parent: Optional[str]
    circuit: Circuit
    topo_hash: str
    design_hash: str
    evaluation: Evaluation
    move: str
    name: str
    n_components: int


class RunRecorder:
    def __init__(self, out_dir: Path, quiet: bool = False):
        self.out = Path(out_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.quiet = quiet
        self.t0 = time.time()
        self._lock = threading.Lock()
        self._progress: dict = {}

    def elapsed(self) -> float:
        return time.time() - self.t0

    # -------------------------------------------------------------- console
    def log(self, phase: str, msg: str) -> None:
        if self.quiet:
            return
        e = int(self.elapsed())
        print(f"[{e // 3600:02d}:{e % 3600 // 60:02d}:{e % 60:02d}] [{phase:<7}] {msg}", flush=True)

    # -------------------------------------------------------------- files
    def write_json(self, name: str, obj) -> None:
        path = self.out / name
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
        os.replace(tmp, path)

    def _append_csv(self, name: str, fields, rows) -> None:
        path = self.out / name
        new = not path.exists()
        with path.open("a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
            if new:
                w.writeheader()
            for r in rows:
                w.writerow(r)

    def event(self, kind: str, **data) -> None:
        with self._lock:
            with (self.out / "events.jsonl").open("a", encoding="utf-8") as f:
                f.write(json.dumps({"t": round(self.elapsed(), 2), "kind": kind, **data}, default=str) + "\n")

    def candidate(self, c: Candidate, split: str = "search") -> None:
        ev = c.evaluation
        with self._lock:
            self._append_csv("candidates.csv", CANDIDATE_FIELDS, [{
                "id": c.id, "iteration": c.iteration, "origin": c.origin, "parent": c.parent or "",
                "move": c.move, "name": c.name, "topo_hash": c.topo_hash, "design_hash": c.design_hash,
                "n_components": c.n_components, "split": split, "scalar": round(ev.scalar, 4),
                "cond_pass": ev.cond_pass, "checks_pass": ev.checks_pass, "all_pass": ev.all_pass}])
            rows = []
            for cond, m in ev.meas.items():
                r = {"candidate": c.id, "split": split, "condition": cond, "sim_ok": ev.sim_ok.get(cond),
                     "sim_error": " ".join(ev.sim_error.get(cond, "").split())[:300]}
                r.update({k: m.get(k) for k in (*MEAS_NAMES, "dv")})
                rows.append(r)
            self._append_csv("measurements.csv", MEAS_FIELDS, rows)
            self._append_csv("checks.csv", CHECK_FIELDS, [
                {"candidate": c.id, "split": split, **{k: v for k, v in asdict(ch).items()}} for ch in ev.checks])
        folder = self.out / "candidates" / c.id
        folder.mkdir(parents=True, exist_ok=True)
        (folder / "circuit.json").write_text(json.dumps(to_dict(c.circuit), indent=1), encoding="utf-8")

    def proposal(self, row: dict) -> None:
        """One row per design the model proposed: its fate (invalid, duplicate, screen_failed, evaluated)."""
        with self._lock:
            self._append_csv("proposals.csv", PROPOSAL_FIELDS, [row])

    def progress(self, **fields) -> None:
        with self._lock:
            self._progress.update(fields)
            self._progress["elapsed_s"] = round(self.elapsed(), 1)
            snapshot = dict(self._progress)
        self.write_json("progress.json", snapshot)

    def llm_exchange(self, iteration: int, attempt: int, messages, reply) -> None:
        self.write_json(f"llm/iter-{iteration:02d}-{attempt}.json",
                        {"iteration": iteration, "attempt": attempt, "messages": messages, "reply": asdict(reply)})

    def timing(self, d: dict) -> None:
        self.write_json("timing.json", d)


In [ ]:
%%writefile piezo_synth/search.py
"""The search loop.

Each round the model proposes up to `proposals_per_round` structures. Every new structure is
  1. screened: net power into a storage node fixed at 1.8 V, short runs, a few value presets
     (a necessary condition for vout_min, see screen.py); failures go back to the model with their powers;
  2. evaluated in full on the search conditions with the best preset's values;
  3. queued for sizing: an evolution strategy (sizer.py) runs generations while the model thinks.
Feedback to the model carries the measurements, the screen powers and physical hints (diagnostics.py).
Held-out conditions and the Monte Carlo robustness check run only after the search has stopped.
"""
from __future__ import annotations

import json
import random
import shutil
import threading
import time
from dataclasses import asdict, dataclass, field
from typing import Dict, List, Optional

from . import spec
from .bench import evaluate_circuit, evaluate_many
from .circuit import Circuit, CircuitError, describe, design_hash, parse_circuit, to_dict, topology_hash
from .diagnostics import codes, dead_v, hints, repeated
from .evaluate import Evaluation, diagnose, is_better
from .llm import LLMReply, proposals_from
from .netlist import build_netlist, reference_circuit
from .prompts import condition_table, iteration_prompt, repair_messages, system_prompt
from .records import Candidate, RunRecorder
from .refine import refine
from .robust import monte_carlo
from .screen import ScreenResult, screen
from .sim import Simulator
from .sizer import Sizer

__all__ = ["SearchConfig", "SearchOutcome", "run_search", "evaluate_circuit"]

SELECTABLE = ("llm", "size", "refine")


@dataclass
class SearchConfig:
    max_rounds: int = 12
    min_rounds: int = 3
    proposals_per_round: int = 3
    wall_clock_s: float = 3600
    max_sims: int = 0                 # simulation budget (0 = unlimited); equal budgets make runs comparable
    stagnation: int = 5               # rounds without improving the best proposed design
    screen: bool = True
    screen_presets: int = 4
    screen_alpha: float = 0.5
    screen_audit: bool = False        # also evaluate and size screen-rejected designs (filter calibration)
    sizer: str = "es"                 # es | coord | none
    size_budget: int = 48             # full evaluations per structure
    size_lambda: int = 8              # designs per sizing generation (simulated in parallel)
    size_min_gens: int = 1            # generations per model wait even if the model answers first
    drain_top: int = 2                # after the last round, finish sizing this many best structures
    diagnostics: bool = True
    hinted_prompt: bool = False       # True: the original prompt with fragments of the textbook answer (prompts.py)
    mc_samples: int = 16              # Monte Carlo samples for the selected design (0 = skip)
    think: bool = True
    seed: int = 1
    continue_after_met: bool = False
    heartbeat_s: float = 15
    model_grace_s: float = 300        # a model call may run this long past wall_clock_s before it is abandoned


@dataclass
class SearchOutcome:
    selected: Candidate
    reference: Candidate
    stop_reason: str
    met_search: bool
    heldout_selected: Evaluation
    heldout_reference: Evaluation
    met_all: bool


@dataclass
class _Topo:
    topo_hash: str
    name: str
    root: Optional[Candidate] = None          # first full evaluation
    screen: Optional[ScreenResult] = None
    sizer: Optional[Sizer] = None
    best: Optional[Candidate] = None
    order: int = 0
    notes: List[str] = field(default_factory=list)

    def sizing_status(self) -> str:
        s = self.sizer
        if s is None:
            return ""
        state = "running" if not s.done else {"spec_met": "finished: spec met", "budget": "finished: budget used",
                                               "converged": "finished: converged",
                                               "no_parameters": "nothing to tune"}.get(s.reason, s.reason)
        return f"{s.n_evals}/{s.budget} evaluations ({state})"


class _Search:
    def __init__(self, cfg: SearchConfig, llm, sim: Simulator, rec: RunRecorder):
        self.cfg, self.llm, self.sim, self.rec = cfg, llm, sim, rec
        self.conds = spec.search_conditions()
        self.rng = random.Random(cfg.seed)
        self.candidates: List[Candidate] = []
        self.by_design: Dict[str, Candidate] = {}
        self.topos: Dict[str, _Topo] = {}
        self.best: Optional[Candidate] = None           # overall, the reference included
        self.best_found: Optional[Candidate] = None     # best proposed (or sized) design
        self.reference: Optional[Candidate] = None
        self.history: List[set] = []
        self.t = {"model_s": 0.0, "model_calls": 0, "prompt_tokens": 0, "completion_tokens": 0,
                  "reasoning_tokens": 0, "size_s": 0.0, "screen_s": 0.0, "model_errors": 0,
                  "n_screen_sims": 0, "n_proposals": 0, "n_screen_pass": 0, "n_invalid": 0, "n_duplicate": 0}
        self.iteration = 0

    # ---------------------------------------------------------------- bookkeeping
    def _new_candidate(self, c: Circuit, ev: Evaluation, origin: str, parent: Optional[str],
                       topo: Optional[_Topo] = None) -> Candidate:
        cand = Candidate(id=f"c{len(self.candidates):04d}", iteration=self.iteration, origin=origin,
                         parent=parent, circuit=c, topo_hash=topology_hash(c), design_hash=design_hash(c),
                         evaluation=ev, move=c.move if origin == "llm" else origin, name=c.name,
                         n_components=len(c.components))
        self.candidates.append(cand)
        self.by_design[cand.design_hash] = cand
        self.rec.candidate(cand)
        self.rec.event("candidate", id=cand.id, origin=origin, parent=parent, name=cand.name,
                       topo_hash=cand.topo_hash, scalar=round(ev.scalar, 4), cond_pass=ev.cond_pass,
                       checks_pass=ev.checks_pass, all_pass=ev.all_pass, n_sims=self.sim.stats["n_sims"])
        if topo is not None and (topo.best is None or is_better(ev, topo.best.evaluation, cand.n_components,
                                                                  topo.best.n_components)):
            topo.best = cand
        if origin in SELECTABLE:
            if self.best_found is None or is_better(ev, self.best_found.evaluation, cand.n_components,
                                                    self.best_found.n_components):
                self.best_found = cand
            if is_better(ev, self.best.evaluation, cand.n_components, self.best.n_components):
                self.best = cand
                self.rec.log("eval", f"new best {cand.id} ({cand.name}): score {ev.scalar:.2f}, "
                                     f"{ev.cond_pass}/{len(self.conds)} conditions pass")
        elif origin == "reference" and self.best is None:
            self.best = cand
        self._progress()
        return cand

    def _progress(self, **extra) -> None:
        b = self.best
        self.rec.progress(iteration=self.iteration, max_rounds=self.cfg.max_rounds,
                          best_id=b.id if b else None, best_name=b.name if b else None,
                          best_scalar=round(b.evaluation.scalar, 3) if b else None,
                          best_cond_pass=b.evaluation.cond_pass if b else None,
                          n_candidates=len(self.candidates), n_sims=self.sim.stats["n_sims"], **extra)

    def _budget_left(self) -> bool:
        if self.rec.elapsed() >= self.cfg.wall_clock_s:
            return False
        return not (self.cfg.max_sims and self.sim.stats["n_sims"] >= self.cfg.max_sims)

    def _budget_reason(self) -> str:
        return "time_budget" if self.rec.elapsed() >= self.cfg.wall_clock_s else "sim_budget"

    def _evaluate(self, c: Circuit, label: str) -> Evaluation:
        self.rec.progress(phase="sim")
        self.rec.log("sim", f"{label}: simulating {len(self.conds)} conditions")
        t = time.time()
        ev = evaluate_circuit(c, self.conds, self.sim, label)
        self.rec.log("eval", f"{label}: score {ev.scalar:.2f}, {ev.cond_pass}/{len(self.conds)} conditions, "
                             f"{ev.checks_pass}/{len(ev.checks)} checks pass ({time.time() - t:.1f} s)")
        return ev

    def _screen(self, c: Circuit, label: str, n_presets: Optional[int] = None) -> ScreenResult:
        self.rec.progress(phase="screen")
        t = time.time()
        before = self.sim.stats["n_sims"]
        r = screen(c, self.sim, self.rng, label, n_presets=n_presets or self.cfg.screen_presets,
                   alpha=self.cfg.screen_alpha)
        self.t["screen_s"] += time.time() - t
        self.t["n_screen_sims"] += self.sim.stats["n_sims"] - before
        self.rec.log("screen", f"{label}: {'PASS' if r.passed else 'fail'}, worst {r.worst_ratio * 100:.0f} % of "
                               f"the power needed ({r.n_presets} presets, {time.time() - t:.1f} s)")
        self.rec.event("screen", label=label, passed=r.passed, worst_ratio=r.worst_ratio, ratio=r.ratio,
                       gain=r.gain, p_net=r.p_net, n_presets=r.n_presets, sim_failed=r.sim_failed)
        return r

    def _account(self, reply: LLMReply) -> None:
        self.t["model_calls"] += 1
        self.t["model_s"] += reply.seconds or 0.0
        for k in ("prompt_tokens", "completion_tokens", "reasoning_tokens"):
            self.t[k] += getattr(reply, k) or 0
        if reply.error and reply.error != "spec_met_during_wait":
            self.t["model_errors"] += 1

    def _hints(self, ev: Optional[Evaluation], scr: Optional[ScreenResult]) -> List[str]:
        return [h.text for h in hints(ev, scr)] if self.cfg.diagnostics else []

    def _entry(self, cand: Candidate, topo: Optional[_Topo] = None) -> dict:
        ev = cand.evaluation
        topo = topo or self.topos.get(cand.topo_hash)
        scr = topo.screen if topo else None
        return {"id": cand.id, "name": cand.name, "topo_hash": cand.topo_hash, "scalar": ev.scalar,
                "cond_pass": ev.cond_pass, "circuit_text": describe(cand.circuit),
                "diagnosis": diagnose(ev), "table": condition_table(ev, self.conds),
                "screen": scr.summary() if scr else "", "sizing": topo.sizing_status() if topo else "",
                "hints": self._hints(ev, scr)}

    def _top(self, k: int = 4) -> List[dict]:
        ranked = sorted((t for t in self.topos.values() if t.best is not None),
                        key=lambda t: -t.best.evaluation.scalar)
        return [self._entry(t.best, t) for t in ranked[:k]]

    def _explored(self) -> List[dict]:
        return [{"topo_hash": t.topo_hash, "name": t.name,
                 "best_scalar": t.best.evaluation.scalar if t.best else float("nan")}
                for t in self.topos.values() if t.best is not None or t.screen is not None]

    # ---------------------------------------------------------------- sizing
    def _size_target(self) -> Optional[_Topo]:
        live = [t for t in self.topos.values() if t.sizer is not None and not t.sizer.done]
        if not live:
            return None
        return max(live, key=lambda t: (t.sizer.best_ev.scalar, t.order))

    def _size_generation(self, topo: _Topo, origin: str = "size") -> bool:
        sz = topo.sizer
        batch = sz.ask()
        if not batch:
            return False
        self.rec.progress(phase="size")
        t = time.time()
        evs = evaluate_many(batch, self.conds, self.sim,
                            [f"{topo.root.id}-size{sz.generations:02d}-{k}" for k in range(len(batch))])
        for c, ev in zip(batch, evs):
            self._new_candidate(c, ev, origin, topo.root.id, topo)
        before = sz.best_ev.scalar
        sz.tell(batch, evs)
        self.t["size_s"] += time.time() - t
        self.rec.log("size", f"{topo.root.id} ({topo.name}) generation {sz.generations}: best score "
                             f"{before:.2f} -> {sz.best_ev.scalar:.2f}, {sz.n_evals}/{sz.budget} evaluations "
                             f"({time.time() - t:.0f} s){'; ' + sz.reason if sz.done else ''}")
        return True

    def _coord(self, topo: _Topo, keep_going) -> None:
        """Ablation: the original coordinate search, run once per structure."""
        def ev_fn(c: Circuit) -> Evaluation:
            dh = design_hash(c)
            if dh in self.by_design:
                return self.by_design[dh].evaluation
            ev = evaluate_circuit(c, self.conds, self.sim, f"{topo.root.id}-coord")
            self._new_candidate(c, ev, "refine", topo.root.id, topo)
            return ev
        t = time.time()
        refine(topo.root.circuit, topo.root.evaluation, ev_fn, self.cfg.size_budget, self.rng,
               should_stop=lambda: not keep_going() or not self._budget_left())
        topo.sizer.done, topo.sizer.reason = True, "budget"
        self.t["size_s"] += time.time() - t

    def _met(self) -> bool:
        """The search has what it was looking for and should stop as soon as possible."""
        return (not self.cfg.continue_after_met and self.best_found is not None
                and self.best_found.evaluation.all_pass)

    def _size_while(self, keep_going) -> None:
        gens = 0
        while self._budget_left() and not self._met():
            if not keep_going() and gens >= self.cfg.size_min_gens:
                return
            topo = self._size_target()
            if topo is None:
                return
            if self.cfg.sizer == "coord":
                self._coord(topo, keep_going)
            elif not self._size_generation(topo):
                continue
            gens += 1

    # ---------------------------------------------------------------- model call
    def _ask(self, messages: List[dict], think: bool, attempt: int) -> LLMReply:
        holder: dict = {}
        deadline = self.cfg.wall_clock_s + self.cfg.model_grace_s      # in run-elapsed seconds
        budget = max(0.0, deadline - self.rec.elapsed())

        def call():
            try:
                holder["r"] = self.llm.ask(messages, think=think, timeout=budget)
            except Exception as e:   # the client should not raise, but never let the thread die silently
                holder["r"] = LLMReply(error=f"{type(e).__name__}: {e}")

        th = threading.Thread(target=call, daemon=True)
        t0 = time.time()
        self.rec.progress(phase="model", model_wait_s=0)
        self.rec.log("model", f"round {self.iteration}: asking the model (thinking {'on' if think else 'off'})")
        th.start()
        if attempt == 1:
            self._size_while(lambda: th.is_alive() and self.rec.elapsed() < deadline)
        last_beat = time.time()
        while th.is_alive():
            if self._met():
                # sizing met the spec while the model was thinking: stop waiting for an answer
                # that can no longer change the result (the daemon thread is abandoned)
                holder.setdefault("r", LLMReply(seconds=round(time.time() - t0, 1), error="spec_met_during_wait"))
                self.rec.log("model", "spec met while waiting; not waiting for the model's answer")
                break
            if self.rec.elapsed() >= deadline:
                holder.setdefault("r", LLMReply(seconds=round(time.time() - t0, 1),
                                                error="model call abandoned: wall-clock budget exceeded"))
                break
            th.join(timeout=min(1.0, self.cfg.heartbeat_s))
            if th.is_alive() and time.time() - last_beat >= self.cfg.heartbeat_s:
                last_beat = time.time()
                waited = time.time() - t0
                self.rec.progress(phase="model", model_wait_s=round(waited))
                self.rec.log("model", f"waiting {waited:.0f} s (round {self.iteration})")
        reply = holder.get("r") or LLMReply(error="model call abandoned: wall-clock budget exceeded")
        self._account(reply)
        self.rec.llm_exchange(self.iteration, attempt, messages, reply)
        tok = (f"{reply.prompt_tokens} in / {reply.completion_tokens} out" if reply.completion_tokens is not None
               else "no token counts")
        self.rec.log("model", f"reply after {reply.seconds:.0f} s ({tok})" + (f"; ERROR {reply.error}" if reply.error else ""))
        self.rec.event("model_reply", iteration=self.iteration, attempt=attempt, seconds=reply.seconds,
                       prompt_tokens=reply.prompt_tokens, completion_tokens=reply.completion_tokens,
                       reasoning_tokens=reply.reasoning_tokens, finish=reply.finish, error=reply.error)
        return reply

    @staticmethod
    def _parse_all(answer: str):
        """[(obj, circuit or None, problems)] and a flat list of problems for a repair request."""
        objs = proposals_from(answer)
        if objs is None:
            return [], ["no JSON object with a \"proposals\" list (or a circuit with \"components\") found"]
        out, flat = [], []
        for k, obj in enumerate(objs, 1):
            try:
                out.append((obj, parse_circuit(obj), []))
            except CircuitError as e:
                out.append((obj, None, e.problems))
                flat += [f"design {k} ({obj.get('name', '?')}): {p}" for p in e.problems]
        if not objs:
            flat.append("the proposals list is empty")
        return out, flat

    # ---------------------------------------------------------------- one proposal
    def _row(self, k: int, name: str, status: str, **extra) -> dict:
        row = {"round": self.iteration, "k": k, "name": name, "status": status, **extra}
        self.rec.proposal(row)
        return row

    def _process(self, k: int, obj: dict, circuit: Optional[Circuit], problems: List[str]) -> dict:
        rec, cfg = self.rec, self.cfg
        name = str(obj.get("name", "")) if isinstance(obj, dict) else ""
        self.t["n_proposals"] += 1
        if circuit is None:
            self.t["n_invalid"] += 1
            rec.event("proposal_invalid", iteration=self.iteration, k=k, problems=problems)
            self._row(k, name, "invalid", problems=" | ".join(problems)[:500])
            return {"kind": "invalid", "name": name, "invalid_problems": problems}
        th = topology_hash(circuit)
        if th in self.topos:
            prev = self.topos[th]
            self.t["n_duplicate"] += 1
            of = prev.root.id if prev.root else prev.name
            rec.event("duplicate", iteration=self.iteration, k=k, of=of, name=circuit.name)
            rec.log("eval", f"round {self.iteration} design {k} '{circuit.name}' repeats structure {of}; not simulated")
            self._row(k, circuit.name, "duplicate", move=circuit.move, topo_hash=th, of=of)
            return {"kind": "duplicate", "name": circuit.name, "id": of}
        topo = _Topo(th, circuit.name, order=len(self.topos))
        self.topos[th] = topo
        rec.event("proposal", iteration=self.iteration, k=k, move=circuit.move, name=circuit.name,
                  rationale=circuit.rationale, circuit=to_dict(circuit))
        rec.log("eval", f"round {self.iteration} design {k}: {circuit.move} '{circuit.name}' "
                        f"({len(circuit.components)} parts)")
        start = circuit
        scr_cols = {}
        if cfg.screen:
            scr = self._screen(circuit, f"r{self.iteration:02d}d{k}")
            topo.screen = scr
            scr_cols = {"worst_ratio": scr.worst_ratio, "n_presets": scr.n_presets,
                        **{f"ratio_{c}": v for c, v in scr.ratio.items()},
                        **{f"gain_{c}": v for c, v in scr.gain.items()}}
            if not scr.passed:
                hs = hints(None, scr)
                self.history.append(codes(hs))
                self._row(k, circuit.name, "screen_failed", move=circuit.move, topo_hash=th,
                          hints=" | ".join(h.code for h in hs), **scr_cols)
                if cfg.screen_audit:
                    self._audit(topo, scr.circuit)
                return {"kind": "screen_failed", "name": circuit.name, "topo_hash": th,
                        "screen": scr.summary(), "hints": self._hints(None, scr),
                        "circuit_text": describe(circuit)}
            self.t["n_screen_pass"] += 1
            start = scr.circuit
        ev = self._evaluate(start, f"r{self.iteration:02d}d{k}")
        cand = self._new_candidate(start, ev, "llm", None, topo)
        topo.root = cand
        hs = hints(ev, topo.screen)
        self.history.append(codes(hs))
        dead = any(h.code == "pinned" for h in hs) or not any(ev.sim_ok.values()) or \
            not any((m.get("vavg") or 0.0) >= dead_v() for m in ev.meas.values())
        if cfg.sizer != "none" and cfg.size_budget > 0 and not ev.all_pass and not dead:
            topo.sizer = Sizer(start, ev, cfg.size_budget, random.Random(self.rng.random()), lam=cfg.size_lambda)
        elif dead:
            rec.log("size", f"not sizing {cand.id} ({cand.name}): output never reaches {dead_v():g} V")
        self._row(k, circuit.name, "evaluated", move=circuit.move, topo_hash=th, candidate=cand.id,
                  hints=" | ".join(h.code for h in hs), **scr_cols)
        return dict(self._entry(cand, topo), kind="evaluated")

    def _audit(self, topo: _Topo, start: Circuit) -> None:
        """Filter calibration: evaluate and size a screen-rejected structure, outside the selection."""
        ev = evaluate_circuit(start, self.conds, self.sim, f"audit-{topo.topo_hash}")
        root = self._new_candidate(start, ev, "audit", None, None)
        sz = Sizer(start, ev, self.cfg.size_budget, random.Random(self.rng.random()), lam=self.cfg.size_lambda)
        tmp = _Topo(topo.topo_hash, topo.name, root=root, sizer=sz)
        while not sz.done and self._budget_left():
            if not self._size_generation(tmp, origin="audit"):
                break
        self.rec.event("audit", topo_hash=topo.topo_hash, name=topo.name, worst_ratio=topo.screen.worst_ratio,
                       met=sz.best_ev.all_pass, best_scalar=sz.best_ev.scalar, n_evals=sz.n_evals)
        self.rec.log("audit", f"screen-rejected '{topo.name}': best after sizing {sz.best_ev.scalar:.2f}, "
                              f"spec met: {sz.best_ev.all_pass}")

    # ---------------------------------------------------------------- main loop
    def run(self) -> SearchOutcome:
        cfg, rec = self.cfg, self.rec
        rec.write_json("spec.json", spec.spec_dict())
        rec.write_json("config.json", asdict(cfg))
        rec.progress(phase="setup", stop_reason=None)
        rec.log("setup", f"reference circuit on {len(self.conds)} search conditions")
        ref_c = reference_circuit()
        ref_c.move = "new_topology"
        self.reference = self._new_candidate(ref_c, self._evaluate(ref_c, "reference"), "reference", None)
        if cfg.screen:
            r = self._screen(ref_c, "reference", n_presets=1)
            rec.write_json("reference_screen.json", {"worst_ratio": r.worst_ratio, "ratio": r.ratio, "gain": r.gain,
                                                     "p_net": r.p_net, "p_req": r.p_req})
        system = system_prompt(self.reference.evaluation, per_round=cfg.proposals_per_round, screen=cfg.screen,
                               size_budget=cfg.size_budget, hinted=cfg.hinted_prompt)
        last: Optional[list] = None
        consecutive_errors = no_improve = 0
        stop_reason = "round_budget"
        for i in range(1, cfg.max_rounds + 1):
            self.iteration = i
            if not self._budget_left():
                stop_reason = self._budget_reason()
                break
            before = self.best_found.evaluation.scalar if self.best_found else None
            rep = repeated(self.history) if cfg.diagnostics else []
            user = iteration_prompt(i, cfg.max_rounds, self._top(), last, self._explored(), None, rep)
            messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
            reply = self._ask(messages, cfg.think, 1)
            done = False
            if self._met():
                stop_reason = "search_spec_met"
                break
            if reply.error:
                consecutive_errors += 1
                last = [{"kind": "error", "error": reply.error}]
                rec.event("model_error", iteration=i, error=reply.error)
            else:
                consecutive_errors = 0
                parsed, problems = self._parse_all(reply.answer)
                if not any(c is not None for _, c, _ in parsed):
                    rec.log("eval", f"round {i}: no usable design ({len(problems)} problems); asking for a correction")
                    rec.event("proposal_rejected", iteration=i, problems=problems)
                    if self._budget_left():
                        reply2 = self._ask(repair_messages(system, user, reply.answer, problems), False, 2)
                        if not reply2.error:
                            parsed2, problems2 = self._parse_all(reply2.answer)
                            if parsed2:
                                parsed, problems = parsed2, problems2
                if not parsed:
                    rec.event("proposal_invalid", iteration=i, problems=problems)
                    last = [{"kind": "invalid", "name": "", "invalid_problems": problems}]
                else:
                    last = []
                    for k, (obj, circuit, probs) in enumerate(parsed[:cfg.proposals_per_round], 1):
                        if not self._budget_left():
                            break
                        done = done or bool(circuit is not None and circuit.done)
                        last.append(self._process(k, obj, circuit, probs))
                        if self.best_found is not None and self.best_found.evaluation.all_pass:
                            break
            after = self.best_found.evaluation.scalar if self.best_found else None
            improved = after is not None and (before is None or after >= before + 0.01)
            no_improve = 0 if improved else no_improve + 1
            self._progress(phase="eval")
            if self.best_found is not None and self.best_found.evaluation.all_pass and not cfg.continue_after_met:
                stop_reason = "search_spec_met"
                break
            if done and i >= cfg.min_rounds:
                stop_reason = "model_done"
                break
            if consecutive_errors >= 3:
                stop_reason = "model_unavailable"
                break
            if no_improve >= cfg.stagnation and i >= cfg.min_rounds:
                stop_reason = "stagnation"
                break
            if not self._budget_left():
                stop_reason = self._budget_reason()
                break
        if stop_reason not in ("search_spec_met", "time_budget", "sim_budget"):
            self._drain()
        return self._finish(stop_reason)

    def _drain(self) -> None:
        """Finish sizing the few best structures once the model is no longer being asked."""
        live = sorted((t for t in self.topos.values() if t.sizer is not None and not t.sizer.done),
                      key=lambda t: -t.sizer.best_ev.scalar)[:self.cfg.drain_top]
        for topo in live:
            self.rec.log("size", f"finishing the sizing of {topo.root.id} ({topo.name})")
            while not topo.sizer.done and self._budget_left():
                if self.cfg.sizer == "coord":
                    self._coord(topo, lambda: True)
                elif not self._size_generation(topo):
                    break
                if self.best_found is not None and self.best_found.evaluation.all_pass:
                    return

    def _finish(self, stop_reason: str) -> SearchOutcome:
        rec = self.rec
        sel, ref = self.best, self.reference
        rec.log("done", f"search stopped: {stop_reason}; selected {sel.id} ({sel.name})")
        rec.event("stop", reason=stop_reason, selected=sel.id)
        held = spec.heldout_conditions()
        rec.progress(phase="heldout", stop_reason=stop_reason)
        rec.log("heldout", f"checking {sel.id} and the reference on {len(held)} held-out conditions")
        ev_sel = evaluate_circuit(sel.circuit, held, self.sim, f"{sel.id}-heldout")
        ev_ref = ev_sel if sel is ref else evaluate_circuit(ref.circuit, held, self.sim, "reference-heldout")
        pairs = [(sel, ev_sel)] if sel is ref else [(sel, ev_sel), (ref, ev_ref)]
        for cand, ev in pairs:
            rec.candidate(Candidate(**{**cand.__dict__, "evaluation": ev}), split="heldout")
        met_search = sel.evaluation.all_pass
        met_all = met_search and ev_sel.all_pass
        rec.log("heldout", f"{sel.id}: {ev_sel.cond_pass}/{len(held)} held-out conditions pass; "
                           f"reference: {ev_ref.cond_pass}/{len(held)}")
        if self.cfg.mc_samples > 0:
            rec.progress(phase="robust")
            rec.log("robust", f"Monte Carlo: {self.cfg.mc_samples} samples of {sel.id} on all {len(spec.CONDITIONS)} "
                              f"conditions")
            mc = monte_carlo(sel.circuit, spec.CONDITIONS, self.sim, random.Random(self.cfg.seed + 7919),
                             self.cfg.mc_samples, f"{sel.id}")
            rec.write_json("robust.json", {"candidate": sel.id, **mc})
            rec.log("robust", f"{mc['pass_all'] * 100:.0f} % of samples meet the spec on every condition; "
                              f"weakest: {mc['worst_condition']}")
        sdir = rec.out / "selected"
        if sdir.exists():
            shutil.rmtree(sdir)
        sdir.mkdir(parents=True)
        (sdir / "circuit.json").write_text(json.dumps(to_dict(sel.circuit), indent=1), encoding="utf-8")
        for cond in spec.CONDITIONS:
            (sdir / f"{cond.id}.net").write_text(build_netlist(sel.circuit, cond, title=sel.id), encoding="utf-8")
        st = self.sim.stats
        timing = {"wall_s": round(rec.elapsed(), 1), "model_s": round(self.t["model_s"], 1),
                  "model_calls": self.t["model_calls"], "model_errors": self.t["model_errors"],
                  "prompt_tokens": self.t["prompt_tokens"], "completion_tokens": self.t["completion_tokens"],
                  "reasoning_tokens": self.t["reasoning_tokens"],
                  "sim_seconds": round(st["sim_seconds"], 1), "sim_wall_s": round(st["wall_seconds"], 1),
                  "n_sims": st["n_sims"], "cache_hits": st["cache_hits"], "sim_failures": st.get("failures", 0),
                  "size_s": round(self.t["size_s"], 1), "screen_s": round(self.t["screen_s"], 1),
                  "n_screen_sims": self.t["n_screen_sims"], "n_proposals": self.t["n_proposals"],
                  "n_invalid": self.t["n_invalid"], "n_duplicate": self.t["n_duplicate"],
                  "n_topologies": len(self.topos), "n_screen_pass": self.t["n_screen_pass"],
                  "rounds": self.iteration,
                  "note": "model calls overlap sizing and simulations run in parallel, so the sums can exceed wall_s"}
        rec.timing(timing)
        failed = [{"split": split, **c.__dict__} for split, ev in (("search", sel.evaluation), ("heldout", ev_sel))
                  for c in ev.checks if not c.passed]
        first_met = next((e for e in self._events_met()), None)
        rec.write_json("outcome.json", {"stop_reason": stop_reason, "met_search": met_search, "met_all": met_all,
                                        "selected_id": sel.id, "selected_name": sel.name,
                                        "selected_origin": sel.origin, "reference_id": ref.id,
                                        "first_met": first_met, "failed_checks": failed})
        rec.progress(phase="done", stop_reason=stop_reason)
        rec.log("done", f"spec met on search conditions: {met_search}; on all conditions: {met_all}")
        return SearchOutcome(sel, ref, stop_reason, met_search, ev_sel, ev_ref, met_all)

    def _events_met(self):
        """When (time, simulations, round) the first spec-meeting proposed design appeared."""
        path = self.rec.out / "events.jsonl"
        if not path.exists():
            return
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                e = json.loads(line)
            except ValueError:
                continue
            if e.get("kind") == "candidate" and e.get("all_pass") and e.get("origin") in SELECTABLE:
                yield {"t": e.get("t"), "n_sims": e.get("n_sims"), "id": e.get("id")}


def run_search(cfg: SearchConfig, llm, sim: Simulator, rec: RunRecorder) -> SearchOutcome:
    return _Search(cfg, llm, sim, rec).run()


In [ ]:
%%writefile piezo_synth/report.py
"""Build report.md and summary.json from a run folder (works on incomplete runs too).

    python -m piezo_synth.report runs/<run_id>
"""
from __future__ import annotations

import csv
import json
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List

from . import spec as spec_mod
from .circuit import describe, parse_circuit


def _read_csv(path: Path, required=()) -> List[dict]:
    """Rows of a CSV; rows cut short by a killed or still-writing job (missing required fields) are skipped."""
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as f:
        return [r for r in csv.DictReader(f) if all(r.get(k) not in (None, "") for k in required)]


def _read_json(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return default


def _events(path: Path) -> List[dict]:
    out = []
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                out.append(json.loads(line))
            except ValueError:
                continue   # a line cut off by a killed job
    return out


def _f(v, fmt="{:.3g}", scale=1.0) -> str:
    if v in (None, ""):
        return "-"
    try:
        return fmt.format(float(v) * scale)
    except ValueError:
        return str(v)


def _table(head: List[str], rows: List[List[str]]) -> str:
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    lines += ["| " + " | ".join(str(c).replace("|", "/") for c in r) + " |" for r in rows]
    return "\n".join(lines)


def _true(v) -> bool:
    return str(v).lower() in ("true", "1")


def build_report(out_dir) -> str:
    out = Path(out_dir)
    spec = _read_json(out / "spec.json") or spec_mod.spec_dict()
    cands = _read_csv(out / "candidates.csv", ("id", "split", "scalar", "n_components", "all_pass"))
    meas = _read_csv(out / "measurements.csv", ("candidate", "condition"))
    checks = _read_csv(out / "checks.csv", ("candidate", "condition", "criterion", "limit", "margin", "passed"))
    events = _events(out / "events.jsonl")
    timing = _read_json(out / "timing.json", {}) or {}
    outcome = _read_json(out / "outcome.json")
    progress = _read_json(out / "progress.json", {}) or {}
    crit_ids = [c["id"] for c in spec["criteria"]]
    conds = spec["conditions"]
    search_ids = [c["id"] for c in conds if c["split"] == "search"]
    held_ids = [c["id"] for c in conds if c["split"] == "heldout"]

    complete = outcome is not None and progress.get("phase") == "done"
    search_rows = [c for c in cands if c["split"] == "search"]
    if outcome:
        sel_id, ref_id = outcome["selected_id"], outcome["reference_id"]
        stop_reason = outcome["stop_reason"] if complete else "incomplete"
    else:
        stop_reason = "incomplete"
        ref_id = next((c["id"] for c in search_rows if c["origin"] == "reference"), None)
        others = [c for c in search_rows if c["origin"] != "reference"] or search_rows
        best = max(others, key=lambda c: (float(c["scalar"]), -int(c["n_components"])), default=None)
        sel_id = best["id"] if best else None
    by_id = {c["id"]: c for c in search_rows}
    sel = by_id.get(sel_id, {})

    M: Dict[tuple, dict] = {(m["candidate"], m["condition"]): m for m in meas}
    K: Dict[tuple, Dict[str, dict]] = defaultdict(dict)
    for k in checks:
        K[(k["candidate"], k["condition"])][k["criterion"]] = k

    def cond_rows(cid: str, ids: List[str]) -> List[List[str]]:
        rows = []
        for cond in ids:
            m = M.get((cid, cond))
            if m is None:
                continue
            ks = K.get((cid, cond), {})
            flags = ["ok" if _true(ks[c]["passed"]) else "**FAIL**" if c in ks else "-" for c in crit_ids]
            if not _true(m.get("sim_ok")):
                rows.append([cond, "sim failed", "", "", "", "", ""] + flags)
                continue
            rows.append([cond, _f(m["vavg"]), _f(m["vmax"]), _f(m["vpp"], scale=1e3), _f(m["tup"]),
                         _f(m["pload"], scale=1e6), _f(m["pext"], scale=1e6)] + flags)
        return rows

    mhead = ["cond", "Vavg V", "Vmax V", "ripple mV", "t_up s", "P_load uW", "P_ext uW"] + crit_ids

    def failed(cid: str, ids: List[str]) -> List[dict]:
        res = []
        for cond in ids:
            for c in crit_ids:
                k = K.get((cid, cond), {}).get(c)
                if k is not None and not _true(k["passed"]):
                    res.append({"condition": cond, "criterion": c, "measured": k["measured"] or None,
                                "op": k["op"], "limit": float(k["limit"]), "margin": float(k["margin"])})
        return res

    def all_pass(cid: str, ids: List[str]) -> bool:
        have = [K.get((cid, c)) for c in ids]
        return all(have) and not failed(cid, ids)

    s = ["# Piezoelectric harvester topology synthesis: run report", ""]
    s.append(f"Run folder: `{out}`. Stop reason: **{stop_reason}**. Selected design: **{sel_id}** "
             f"({sel.get('name', '?')}).")
    # 1. specification
    s += ["", "## Specification", "",
          "Every criterion must hold under every operating condition (0.8 s cold-start transient).", ""]
    s.append(_table(["criterion", "measurement", "requirement", "meaning"],
                    [[c["id"], c["meas"], f"{c['op']} {c['limit']:g} {c['unit']}", c["description"]]
                     for c in spec["criteria"]]))
    lim = spec["limits"]
    s += ["", f"Limits: at most {lim['max_components']} components, {lim['max_switches']} switches, "
              f"{lim['max_sync']} SYNC controllers (each draws {spec.get('ctrl_sink_a', 0.5e-6) * 1e6:g} uA from out); "
              f"L {lim['L'][0]:g}-{lim['L'][1]:g} H with series resistance {lim['l_rser_base']:g} Ohm + "
              f"{lim['l_rser_per_h'] / 1000:g} Ohm/mH; C up to {lim['C'][1]:g} F."]
    # 2. conditions
    s += ["", "## Test conditions", ""]
    s.append(_table(["id", "set", "description", "Ip uA pk", "f Hz", "Cp nF", "RL kOhm", "T C"],
                    [[c["id"], "search" if c["split"] == "search" else "held-out", c["label"], _f(c["ip"], scale=1e6),
                      _f(c["freq"]), _f(c["cp"], scale=1e9), _f(c["rl"], scale=1e-3), _f(c["temp"])] for c in conds]))
    s += ["", "Held-out conditions were never shown to the model and were not used for scoring, selection or stopping."]
    # 3. reference
    s += ["", "## Reference results", "",
          f"Reference: full-bridge rectifier, 4 x Schottky, 2.2 uF (candidate {ref_id})."]
    s += ["", _table(mhead, cond_rows(ref_id, search_ids + held_ids))] if ref_id else ["", "(not evaluated)"]
    # 4. proposals
    s += ["", "## Proposals tried", ""]
    props = _read_csv(out / "proposals.csv", ("round", "k", "status"))
    per_round: Dict[int, dict] = defaultdict(lambda: {"s": 0.0, "tok": 0, "calls": 0})
    for e in events:
        if e["kind"] == "model_reply":
            d = per_round[e["iteration"]]
            d["s"] += e.get("seconds") or 0
            d["tok"] += (e.get("completion_tokens") or 0)
            d["calls"] += 1
    screen_ids = list(spec.get("screen", {}).get("conditions", []))
    rows = []
    for p in props:
        c = by_id.get(p.get("candidate") or "", {})
        ratios = " ".join(f"{cid} {float(p['ratio_' + cid]) * 100:.0f} %" for cid in screen_ids
                          if p.get("ratio_" + cid) not in (None, "", "-inf"))
        if p["status"] == "invalid":
            detail = (p.get("problems") or "")[:120]
        elif p["status"] == "duplicate":
            detail = f"same as {p.get('of')}"
        else:
            detail = ratios
        rows.append([p["round"], p["k"], p.get("name", ""), p.get("move", ""), p.get("topo_hash", ""), p["status"],
                     detail, p.get("candidate") or "-",
                     f"{c['cond_pass']}/{len(search_ids)}" if c else "-", _f(c.get("scalar"), "{:.2f}") if c else "-",
                     p.get("hints") or "-"])
    s.append(_table(["round", "#", "name", "move", "topology", "outcome", "screen (power vs need) / detail",
                     "candidate", "cond. pass", "score", "problems found"], rows) if rows else "(none)")
    errors = [e for e in events if e["kind"] == "model_error"]
    if per_round:
        s += ["", "Model calls per round:", "",
              _table(["round", "calls", "model s", "completion tokens"],
                     [[r, d["calls"], f"{d['s']:.0f}", d["tok"] or "-"] for r, d in sorted(per_round.items())])]
    if errors:
        s += ["", f"{len(errors)} model call(s) failed: " + "; ".join(str(e.get("error"))[:80] for e in errors[:3])]
    rscreen = _read_json(out / "reference_screen.json")
    if rscreen:
        s += ["", f"Reference on the screen (power delivered at {spec.get('screen', {}).get('vstore', 1.8):g} V vs need): " +
              ", ".join(f"{k} {v * 100:.0f} %" for k, v in sorted(rscreen.get("ratio", {}).items())) + "."]
    # 4b. sizing
    s += ["", "## Value sizing", ""]
    tuned = [c for c in search_rows if c["origin"] in ("size", "refine")]
    if tuned:
        grp: Dict[str, List[dict]] = defaultdict(list)
        for c in tuned:
            grp[c["parent"]].append(c)
        trows = []
        for p, v in grp.items():
            top = max(v, key=lambda x: float(x["scalar"]))
            trows.append([p, by_id.get(p, {}).get("name", ""), len(v), _f(by_id.get(p, {}).get("scalar"), "{:.2f}"),
                          _f(top["scalar"], "{:.2f}"), top["id"], f"{top['cond_pass']}/{len(search_ids)}"])
        s.append(_table(["structure (first evaluation)", "name", "evaluations", "start score", "best score",
                         "best candidate", "conditions passed"], trows))
    else:
        s.append("No structure was sized (none passed the screen with an output to tune, or sizing was off).")
    audits = [e for e in events if e["kind"] == "audit"]
    if audits:
        s += ["", "Screen audit (structures rejected by the screen, sized anyway to check the filter):", "",
              _table(["structure", "screen worst %", "evaluations", "best score", "spec met"],
                     [[a.get("name"), f"{(a.get('worst_ratio') or 0) * 100:.0f}", a.get("n_evals"),
                       _f(a.get("best_scalar"), "{:.2f}"), a.get("met")] for a in audits])]
    ntopo = len({c["topo_hash"] for c in search_rows if c["origin"] not in ("reference", "audit")})
    s += ["", f"{len(search_rows)} designs evaluated on the search conditions, {ntopo} distinct structures "
              f"(excluding the reference)."]
    # 5. selected design
    s += ["", "## Selected design", ""]
    circ = _read_json(out / "candidates" / str(sel_id) / "circuit.json") if sel_id else None
    if circ:
        try:
            text = describe(parse_circuit(circ))
        except Exception:   # an old or hand-edited file: show it raw
            text = json.dumps(circ, indent=1)
        s += [f"**{sel_id}**: {sel.get('name', '')} (origin {sel.get('origin', '?')}, topology {sel.get('topo_hash', '?')})",
              "", "```", text, "```", ""]
    s.append(_table(mhead, cond_rows(sel_id, search_ids)) if sel_id else "(none)")
    # 6. held-out
    s += ["", "## Held-out check", ""]
    if any((sel_id, c) in M for c in held_ids):
        s.append("Selected design:")
        s += ["", _table(mhead, cond_rows(sel_id, held_ids)), ""]
        if ref_id and ref_id != sel_id:
            s += ["Reference:", "", _table(mhead, cond_rows(ref_id, held_ids))]
    else:
        s.append("Not run (the search did not finish).")
    # 6b. robustness
    s += ["", "## Robustness", ""]
    rob = _read_json(out / "robust.json")
    if rob:
        tol = rob.get("tolerances", {})
        s.append(f"Monte Carlo on {rob.get('candidate')}: {rob['n']} samples, values drawn uniformly within "
                 f"R +/-{tol.get('R', 0) * 100:g} %, C +/-{tol.get('C', 0) * 100:g} %, L +/-{tol.get('L', 0) * 100:g} %, "
                 f"switch on-time +/-{tol.get('ton', 0) * 100:g} %, switch timing +/-{tol.get('phase', 0) * 100:g} % "
                 f"of a period. **{rob['pass_all'] * 100:.0f} %** of samples meet every criterion on every condition.")
        s += ["", _table(["condition", "samples passing"],
                         [[k, f"{v * 100:.0f} %"] for k, v in rob.get("per_condition", {}).items()])]
    else:
        s.append("Not run.")
    # 7. unmet
    fails = [dict(f, split="search") for f in failed(sel_id, search_ids)] + \
            [dict(f, split="heldout") for f in failed(sel_id, held_ids)] if sel_id else []
    s += ["", "## Unmet requirements", ""]
    if fails:
        s.append(_table(["set", "condition", "criterion", "measured", "requirement", "margin"],
                        [[f["split"], f["condition"], f["criterion"],
                          _f(f["measured"]) if f["measured"] else "not reached", f"{f['op']} {f['limit']:g}",
                          f"{f['margin']:+.2f}"] for f in fails]))
    else:
        s.append("None: the selected design meets every criterion on every evaluated condition.")
    met_search = bool(sel_id) and all_pass(sel_id, search_ids)
    met_all = met_search and all_pass(sel_id, held_ids)
    # 8. stop reason
    explain = {
        "search_spec_met": "a design met every criterion on every search condition",
        "model_done": "the model declared that no further structural change would help",
        "model_unavailable": "three model calls in a row failed",
        "stagnation": "the best score stopped improving",
        "time_budget": "the wall-clock budget was used up",
        "proposal_budget": "the maximum number of proposals was reached",
        "round_budget": "the maximum number of rounds was reached",
        "sim_budget": "the simulation budget was used up",
        "smoke": "smoke test (reference and witness only, no model)",
        "incomplete": "the job ended before the search finished (killed, timed out, or still running)",
    }
    s += ["", "## Stop reason", "", f"**{stop_reason}**: {explain.get(stop_reason, '')}.",
          f"Spec met on search conditions: **{met_search}**; on all conditions including held-out: **{met_all}**."]
    # 9. time
    s += ["", "## Time and tokens", ""]
    if timing:
        s.append(_table(["quantity", "value"], [
            ["wall time", f"{timing.get('wall_s', 0):.0f} s"],
            ["model calls", timing.get("model_calls")], ["time in model calls", f"{timing.get('model_s', 0):.0f} s"],
            ["prompt tokens", timing.get("prompt_tokens")], ["completion tokens", timing.get("completion_tokens")],
            ["of which reasoning tokens", timing.get("reasoning_tokens") or "not reported"],
            ["simulations run", timing.get("n_sims")], ["cache hits (not re-simulated)", timing.get("cache_hits")],
            ["simulator time (sum over parallel runs)", f"{timing.get('sim_seconds', 0):.0f} s"],
            ["simulation wall time", f"{timing.get('sim_wall_s', 0):.0f} s"],
            ["screening simulations", timing.get("n_screen_sims", "-")],
            ["screening wall time", f"{timing.get('screen_s', 0):.0f} s"],
            ["value sizing wall time", f"{timing.get('size_s', timing.get('refine_s', 0)):.0f} s"],
            ["designs proposed / invalid / duplicate", f"{timing.get('n_proposals', '-')} / "
             f"{timing.get('n_invalid', '-')} / {timing.get('n_duplicate', '-')}"],
            ["structures / passed the screen", f"{timing.get('n_topologies', '-')} / {timing.get('n_screen_pass', '-')}"]]))
        s += ["", "Value sizing runs while the model is thinking, so model and simulation times overlap."]
    else:
        s.append("Not available (run incomplete).")
    # 10. files
    s += ["", "## Files", "", _table(["file", "contents"], [
        ["selected/circuit.json, selected/<cond>.net", "selected design and its exact netlist for every condition"],
        ["proposals.csv", "every design the model proposed and what happened to it (screen powers)"],
        ["candidates.csv", "every design evaluated (one row per design and condition set)"],
        ["robust.json, reference_screen.json", "Monte Carlo result; the reference's screen powers"],
        ["measurements.csv", "every measurement, per design and condition"],
        ["checks.csv", "every pass/fail decision with margin"],
        ["candidates/<id>/circuit.json", "each design's circuit"],
        ["llm/iter-NN-a.json", "prompts, replies and model reasoning (NN = round)"],
        ["events.jsonl", "timeline of the run"], ["sims/<hash>/", "LTspice netlists, logs and waveforms"],
        ["timing.json, outcome.json, spec.json, config.json", "timing, result, spec and settings"]])]
    text = "\n".join(s) + "\n"
    (out / "report.md").write_text(text, encoding="utf-8")
    summary = {"stop_reason": stop_reason, "met_search": met_search, "met_all": met_all, "selected_id": sel_id,
               "selected_topology": {"name": sel.get("name"), "topo_hash": sel.get("topo_hash")},
               "reference_id": ref_id, "failed_checks": fails, "timing": timing}
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return text


if __name__ == "__main__":
    print(build_report(sys.argv[1] if len(sys.argv) > 1 else "."))


In [ ]:
%%writefile piezo_synth/experiments.py
"""Aggregate many runs (variants x seeds) and write the paper's tables.

    python -m piezo_synth.experiments runs/exp --paper paper/generated --smoke runs/smoke

Every run folder below the root that has outcome.json counts; its variant is the --label it was run
with (run_meta.json), else the name of its parent folder. Tables for which no data exist say
"pending" instead of showing numbers.
"""
from __future__ import annotations

import argparse
import csv
import json
import statistics
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from . import spec

PENDING = r"\textit{pending: no runs yet}"

# The paper's comparison: config overrides and extra command-line arguments per variant. Every
# variant shares the base config (and with it the simulation budget max_sims).
VARIANTS: Dict[str, Tuple[dict, str]] = {
    "full": ({}, ""),
    "no_screen": ({"screen": False}, ""),
    "no_diag": ({"diagnostics": False}, ""),
    "single": ({"proposals_per_round": 1, "max_rounds": 24}, ""),
    "coord": ({"sizer": "coord"}, ""),
    "no_think": ({"think": False}, ""),
    # stopped only by the simulation budget or the wall clock, like a model run that never succeeds
    "random": ({"stagnation": 10 ** 6, "max_rounds": 10 ** 4}, "--baseline random"),
    "audit": ({"screen_audit": True}, ""),
    # prompt-leakage control: the original prompt, whose JSON example is the SSHI branch and which gives
    # the switch-timing formula and the bridge and zener-clamp wiring (every other variant is neutral)
    "hinted": ({"hinted_prompt": True}, ""),
}
CAMPAIGN_BASE = {"max_sims": 3000, "wall_clock_s": 3600, "mc_samples": 16}


def write_campaign_configs(root, base: Optional[dict] = None) -> Dict[str, Path]:
    """runs/exp/<variant>.json for every variant; returns their paths."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    out = {}
    for name, (over, _extra) in VARIANTS.items():
        p = root / f"{name}.json"
        p.write_text(json.dumps(dict(CAMPAIGN_BASE if base is None else base, **over), indent=1), encoding="utf-8")
        out[name] = p
    return out


def campaign_loop(name: str, seeds, root: str = "runs/exp") -> str:
    """Shell command running one variant's seeds in turn, skipping seeds already finished.
    Contains no single quotes, so it can be wrapped in bash -c '...'."""
    extra = VARIANTS[name][1]
    return " ; ".join(
        f"( test -f {root}/{name}/seed{s}/outcome.json || python3 -u -m piezo_synth.run --config {root}/{name}.json "
        f"--label {name} --seed {s} {extra} --out {root}/{name}/seed{s} > {root}/{name}-seed{s}.log 2>&1 )"
        for s in seeds)

# The benchmark: every spec of spec.SPECS with these variants (one cluster job per spec).
BENCH_VARIANTS = ["full", "no_screen", "no_diag", "coord", "random"]
BENCH_EXTRA = ["hinted"]      # run on some specs only (tools/panda_campaign.py --bench --variants ...)


def bench_loop(spec_name: str, variants, seeds, root: str = "runs/bench") -> str:
    """Shell command running one spec's variants and seeds in turn, skipping runs already finished.
    The variant configs are root/<variant>.json (write_campaign_configs(root)). No single quotes."""
    parts = []
    for v in variants:
        extra = VARIANTS[v][1]
        for s in seeds:
            d = f"{root}/{spec_name}/{v}/seed{s}"
            parts.append(f"( test -f {d}/outcome.json || python3 -u -m piezo_synth.run --spec {spec_name} "
                         f"--config {root}/{v}.json --label {v} --seed {s} {extra} --out {d} "
                         f"> {root}/{spec_name}-{v}-seed{s}.log 2>&1 )")
    return " ; ".join(parts)


RUN_FIELDS = ["spec", "variant", "seed", "run", "stop_reason", "met_search", "met_all", "rounds", "n_proposals",
              "n_invalid", "n_duplicate", "n_topologies", "n_screen_pass", "n_sims", "n_screen_sims", "wall_s",
              "model_s", "model_calls", "prompt_tokens", "completion_tokens", "first_met_t", "first_met_sims",
              "heldout_pass", "mc_pass_all", "selected_name", "selected_origin"]


def _json(p: Path, default=None):
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return default


def _heldout_pass(run: Path, cid: str) -> Optional[int]:
    path = run / "candidates.csv"
    if not path.exists():
        return None
    with path.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r.get("id") == cid and r.get("split") == "heldout":
                return int(r["cond_pass"])
    return None


def _run_row(run: Path) -> Optional[dict]:
    outcome = _json(run / "outcome.json")
    if not outcome or outcome.get("stop_reason") == "smoke":
        return None
    meta = _json(run / "run_meta.json", {}) or {}
    cfg = _json(run / "config.json", {}) or {}
    t = _json(run / "timing.json", {}) or {}
    rob = _json(run / "robust.json")
    fm = outcome.get("first_met") or {}
    row = {"spec": meta.get("spec") or "P1", "variant": meta.get("label") or run.parent.name, "seed": meta.get("seed", cfg.get("seed")),
           "run": str(run), "stop_reason": outcome["stop_reason"], "met_search": bool(outcome["met_search"]),
           "met_all": bool(outcome["met_all"]), "first_met_t": fm.get("t"), "first_met_sims": fm.get("n_sims"),
           "heldout_pass": _heldout_pass(run, outcome["selected_id"]),
           "mc_pass_all": rob.get("pass_all") if rob else None,
           "selected_name": outcome.get("selected_name"), "selected_origin": outcome.get("selected_origin")}
    for k in ("rounds", "n_proposals", "n_invalid", "n_duplicate", "n_topologies", "n_screen_pass", "n_sims",
              "n_screen_sims", "wall_s", "model_s", "model_calls", "prompt_tokens", "completion_tokens"):
        row[k] = t.get(k)
    return row


def _median(xs) -> Optional[float]:
    xs = [x for x in xs if x is not None]
    return statistics.median(xs) if xs else None


def _min(xs):
    xs = [x for x in xs if x is not None]
    return min(xs) if xs else None


def _max(xs):
    xs = [x for x in xs if x is not None]
    return max(xs) if xs else None


def fisher_two_sided(a: int, n_a: int, b: int, n_b: int) -> float:
    """Two-sided Fisher exact test for success counts a of n_a against b of n_b."""
    from math import comb
    k, n = a + b, n_a + n_b
    total = comb(n, k)
    def p(x):
        return comb(n_a, x) * comb(n_b, k - x) / total
    obs = p(a)
    lo, hi = max(0, k - n_b), min(k, n_a)
    return min(1.0, sum(p(x) for x in range(lo, hi + 1) if p(x) <= obs * (1 + 1e-9)))


def _mean(xs) -> Optional[float]:
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else None


def _audits(runs: List[Path]) -> Tuple[int, int]:
    """(screen-rejected structures audited, of which sizing still met the spec)."""
    n = met = 0
    for run in runs:
        p = run / "events.jsonl"
        if not p.exists():
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            try:
                e = json.loads(line)
            except ValueError:
                continue
            if e.get("kind") == "audit":
                n += 1
                met += bool(e.get("met"))
    return n, met


def aggregate(root) -> Tuple[List[dict], List[dict]]:
    root = Path(root)
    run_dirs = sorted(p.parent for p in root.rglob("outcome.json")) if root.exists() else []
    runs = [r for r in (_run_row(d) for d in run_dirs) if r]
    by: Dict[str, List[dict]] = {}
    for r in runs:
        by.setdefault((r["spec"], r["variant"]), []).append(r)
    summary = []
    for (sp, v), rs in sorted(by.items()):
        props = sum(r["n_proposals"] or 0 for r in rs)
        summary.append({
            "spec": sp, "variant": v, "runs": len(rs),
            "success_rate": sum(r["met_search"] for r in rs) / len(rs),
            "met_all_rate": sum(r["met_all"] for r in rs) / len(rs),
            "median_sims_to_success": _median(r["first_met_sims"] for r in rs if r["met_search"]),
            "min_sims_to_success": _min(r["first_met_sims"] for r in rs if r["met_search"]),
            "max_sims_to_success": _max(r["first_met_sims"] for r in rs if r["met_search"]),
            "median_time_to_success_s": _median(r["first_met_t"] for r in rs if r["met_search"]),
            "median_sims": _median(r["n_sims"] for r in rs),
            "median_wall_s": _median(r["wall_s"] for r in rs),
            "median_model_s": _median(r["model_s"] for r in rs),
            "median_model_calls": _median(r["model_calls"] for r in rs),
            "successes": sum(r["met_search"] for r in rs),
            "successes_all": sum(r["met_all"] for r in rs),
            "mean_completion_tokens": _mean(r["completion_tokens"] for r in rs),
            "invalid_rate": (sum(r["n_invalid"] or 0 for r in rs) / props) if props else None,
            "screen_pass_rate": (sum(r["n_screen_pass"] or 0 for r in rs) /
                                 max(1, sum(r["n_topologies"] or 0 for r in rs))),
            "median_mc_pass": _median(r["mc_pass_all"] for r in rs),
            "mean_heldout_pass": _mean(r["heldout_pass"] for r in rs),
        })
    if root.exists():
        with (root / "experiments.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=RUN_FIELDS, extrasaction="ignore")
            w.writeheader()
            w.writerows(runs)
        if summary:
            with (root / "experiments_summary.csv").open("w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=list(summary[0]))
                w.writeheader()
                w.writerows(summary)
    return runs, summary


# ---------------------------------------------------------------- LaTeX

def tex(s) -> str:
    s = str(s)
    for a, b in (("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"), ("#", r"\#"),
                 ("_", r"\_"), ("{", r"\{"), ("}", r"\}"), ("~", r"\textasciitilde{}"), ("^", r"\^{}")):
        s = s.replace(a, b)
    s = s.replace("−", "$-$").replace("µ", r"$\mu$").replace("°", r"$^\circ$")
    return s.encode("ascii", "replace").decode("ascii")   # pdflatex-safe whatever else a label holds


def _num(v, fmt="{:.0f}", scale=1.0) -> str:
    return "--" if v is None else fmt.format(v * scale)


def _tabular(cols: str, head: List[str], rows: List[List[str]], ncols: int) -> str:
    lines = [r"\begin{tabular}{" + cols + "}", r"\toprule", " & ".join(head) + r" \\", r"\midrule"]
    if rows:
        lines += [" & ".join(r) + r" \\" for r in rows]
    else:
        lines.append(r"\multicolumn{" + str(ncols) + "}{c}{" + PENDING + r"} \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines) + "\n"


def _conditions_table() -> str:
    rows = [[c.id, "search" if c.split == "search" else "held-out", tex(c.label), f"{c.ip * 1e6:g}", f"{c.freq:g}",
             f"{c.cp * 1e9:g}", f"{c.rl / 1e3:g}", f"{c.temp:g}"] for c in spec.CONDITIONS]
    return _tabular("llp{3.2cm}rrrrr", ["ID", "Set", "Description", r"$I_p$ ($\mu$A)", "$f$ (Hz)", "$C_p$ (nF)",
                                        r"$R_L$ (k$\Omega$)", r"$T$ ($^\circ$C)"], rows, 8)


def _criteria_table() -> str:
    sym = {">=": r"$\geq$", "<=": r"$\leq$"}
    rows = [[tex(c.id), tex(c.meas), f"{sym[c.op]} {c.limit:g} {tex(c.unit)}", tex(c.description)] for c in spec.CRITERIA]
    return _tabular("llrp{5.2cm}", ["Criterion", "Measure", "Requirement", "Meaning"], rows, 4)


VARIANT_ORDER = ["full", "no_screen", "no_diag", "single", "coord", "no_think", "hinted", "random"]


def _variants_table(summary: List[dict]) -> str:
    order = {v: k for k, v in enumerate(VARIANT_ORDER)}
    rows = []
    for s in sorted(summary, key=lambda s: order.get(s["variant"], 99)):
        rows.append([tex(s["variant"]), str(s["runs"]), _num(s["success_rate"], "{:.0f}\\%", 100),
                     _num(s["met_all_rate"], "{:.0f}\\%", 100), _range(s.get("median_sims_to_success"), s.get("min_sims_to_success"),
                                                          s.get("max_sims_to_success")),
                     _num(s["median_time_to_success_s"], "{:.0f}", 1 / 60), _num(s["median_sims"]),
                     _num(s["invalid_rate"], "{:.0f}\\%", 100), _num(s["median_mc_pass"], "{:.0f}\\%", 100)])
    return _tabular("lrrrrrrrr", ["Variant", "Runs", "Spec met", "+held-out", "Sims to success",
                                  "Min to success", "Sims", "Invalid", "MC pass"], rows, 9)


def _range(med, lo, hi) -> str:
    if med is None:
        return "--"
    if lo is None or hi is None or lo == hi:
        return _num(med)
    return f"{_num(med)} [{_num(lo)}--{_num(hi)}]"


def _audit_table(n: int, met: int) -> str:
    rows = [[str(n), str(met), _num((n - met) / n if n else None, "{:.0f}\\%", 100)]] if n else []
    return _tabular("rrr", ["Rejected and audited", "Met spec after sizing", "Correct rejections"], rows, 3)


def _cost_table(summary: List[dict]) -> str:
    order = {v: k for k, v in enumerate(VARIANT_ORDER)}
    rows = []
    for s in sorted(summary, key=lambda s: order.get(s["variant"], 99)):
        rows.append([tex(s["variant"]), _num(s.get("median_wall_s"), "{:.1f}", 1 / 60),
                     _num(s.get("median_model_s"), "{:.1f}", 1 / 60), _num(s.get("median_model_calls")),
                     _num(s.get("mean_completion_tokens"), "{:.1f}", 1e-3), _num(s.get("median_sims"))])
    return _tabular("lrrrrr", ["Variant", "Wall (min)", "Model (min)", "Model calls", "Completion tokens (k)",
                               "Sims"], rows, 6)


def _reference_table(smoke: Optional[Path]) -> str:
    """Reference (c0000) and witness (c0001) per condition, from a smoke-test run folder."""
    rows = []
    if smoke and (smoke / "measurements.csv").exists():
        with (smoke / "measurements.csv").open(newline="", encoding="utf-8") as f:
            meas = {(r["candidate"], r["condition"]): r for r in csv.DictReader(f)}
        passed: Dict[tuple, bool] = {}
        with (smoke / "checks.csv").open(newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                k = (r["candidate"], r["condition"])
                passed[k] = passed.get(k, True) and r["passed"].lower() == "true"
        for c in spec.CONDITIONS:
            cells = [c.id]
            for cid in ("c0000", "c0001"):
                m = meas.get((cid, c.id))
                if m is None or m.get("sim_ok", "").lower() != "true":
                    cells += ["--", "--", "--"]
                    continue
                v = float(m["vavg"]) if m.get("vavg") else None
                vmax = float(m["vmax"]) if m.get("vmax") else None
                cells += [_num(v, "{:.2f}"), _num(vmax, "{:.2f}"), "pass" if passed.get((cid, c.id)) else "fail"]
            rows.append(cells)
    return _tabular("l rrl rrl", ["Cond.", r"FBR $\bar V$", r"FBR $V_{max}$", "FBR", r"Witness $\bar V$",
                                   r"Witness $V_{max}$", "Witness"], rows, 7)


def _screen_validation_table(smoke: Optional[Path]) -> str:
    """Screen prediction (net power at 1.8 V vs the load's need) against the full 0.8 s simulation,
    for the reference (c0000) and the witness (c0001) on the screen conditions."""
    rows = []
    scr = _json(smoke / "smoke_screen.json") if smoke else None
    if scr and (smoke / "measurements.csv").exists():
        with (smoke / "measurements.csv").open(newline="", encoding="utf-8") as f:
            vavg = {(r["candidate"], r["condition"]): r.get("vavg") for r in csv.DictReader(f)
                    if r.get("split", "search") == "search"}
        vmin = next(c.limit for c in spec.CRITERIA if c.id == "vout_min")
        for cid, origin, label in (("c0000", "reference", "FBR"), ("c0001", "witness", "Witness")):
            for cond in sorted(scr[origin]["ratio"]):
                ratio = scr[origin]["ratio"][cond]
                pn = scr[origin]["p_net"].get(cond)
                gain = scr[origin]["gain"].get(cond)
                v = vavg.get((cid, cond))
                v = float(v) if v not in (None, "") else None
                rows.append([label, cond, _num(pn, "{:.1f}", 1e6), _num(scr[origin]["p_req"][cond], "{:.1f}", 1e6),
                             _num(ratio, "{:.0f}\\%", 100), _num(gain, "{:.2f}"),
                             r"$\geq$1.8 V" if ratio >= 1 else r"$<$1.8 V",
                             _num(v, "{:.2f} V"), "yes" if (v is not None and (v >= vmin) == (ratio >= 1)) else "no"])
    return _tabular("llrrrrlll", ["Circuit", "Cond.", r"$P_{net}$ ($\mu$W)", r"Need ($\mu$W)", "Ratio",
                                  r"$P_{net}/P^\star$", "Predicted", r"Simulated $\bar V$", "Agrees"], rows, 9)


def _macros(summary: List[dict], runs: List[dict], audit: Tuple[int, int]) -> str:
    def m(name, val):
        return "\\newcommand{\\" + name + "}{" + (val if val is not None else r"\textbf{[pending]}") + "}\n"
    full = next((s for s in summary if s["variant"] == "full"), None)
    rnd = next((s for s in summary if s["variant"] == "random"), None)
    out = m("NRuns", str(len(runs)) if runs else None)
    out += m("FullSuccess", _num(full["success_rate"], "{:.0f}\\%", 100) if full else None)
    out += m("RandomSuccess", _num(rnd["success_rate"], "{:.0f}\\%", 100) if rnd else None)
    out += m("FullSimsToSuccess", _num(full["median_sims_to_success"]) if full else None)
    out += m("RandomMedianSims", _num(rnd["median_sims"]) if rnd else None)
    if full and rnd:
        out += m("RandomVsFullP", "{:.3f}".format(fisher_two_sided(full["successes"], full["runs"],
                                                                    rnd["successes"], rnd["runs"])))
    else:
        out += m("RandomVsFullP", None)
    model = [s for s in summary if s["variant"] not in ("random",)]
    pairs = [fisher_two_sided(a[k], a["runs"], b[k], b["runs"])
             for i, a in enumerate(model) for b in model[i + 1:] for k in ("successes", "successes_all")]
    out += m("MinPairP", "{:.2f}".format(min(pairs)) if pairs else None)
    out += m("AuditN", str(audit[0]) if audit[0] else None)
    out += m("AuditMet", str(audit[1]) if audit[0] else None)
    return out


def write_paper_tables(gen_dir, root, smoke=None) -> None:
    gen = Path(gen_dir)
    gen.mkdir(parents=True, exist_ok=True)
    root = Path(root)
    runs, summary = aggregate(root)
    runs = [r for r in runs if r["spec"] == "P1"]
    summary = [s for s in summary if s["spec"] == "P1"]
    run_dirs = sorted(p.parent for p in root.rglob("outcome.json")) if root.exists() else []
    audit = _audits(run_dirs)
    files = {"tab_conditions.tex": _conditions_table(), "tab_criteria.tex": _criteria_table(),
             "tab_variants.tex": _variants_table(summary), "tab_screen_audit.tex": _audit_table(*audit),
             "tab_cost.tex": _cost_table(summary),
             "tab_reference.tex": _reference_table(Path(smoke) if smoke else None),
             "tab_screen_validation.tex": _screen_validation_table(Path(smoke) if smoke else None),
             "macros.tex": _macros(summary, runs, audit)}
    for name, text in files.items():
        (gen / name).write_text(text, encoding="utf-8")


def _bench_table(summary: List[dict]) -> str:
    """Specs as rows, variants as columns: runs meeting the search spec / runs (and also held-out)."""
    specs = [sp for sp in spec.SPECS if any(s["spec"] == sp for s in summary)]
    variants = [v for v in BENCH_VARIANTS + BENCH_EXTRA if any(s["variant"] == v for s in summary)]
    cell = {(s["spec"], s["variant"]): s for s in summary}
    rows = []
    for sp in specs:
        row = [sp, tex(spec.SPECS[sp].title)]
        for v in variants:
            c = cell.get((sp, v))
            if c is None:
                row.append("--")
                continue
            med = c.get("median_sims_to_success")
            row.append(f"{c['successes']}/{c['runs']} ({c['successes_all']})"
                       + (f", {med:.0f}" if med is not None else ""))
        rows.append(row)
    return _tabular("ll" + "r" * len(variants), ["Spec", "Requirement"] + [tex(v) for v in variants], rows,
                    2 + max(1, len(variants)))


def write_bench_tables(gen_dir, root) -> None:
    gen = Path(gen_dir)
    gen.mkdir(parents=True, exist_ok=True)
    runs, summary = aggregate(Path(root))

    def m(name, val):
        return "\\newcommand{\\" + name + "}{" + (val if val is not None else r"\textbf{[pending]}") + "}\n"

    model = [s for s in summary if s["variant"] != "random"]
    rnd = [s for s in summary if s["variant"] == "random"]
    out = m("BenchRuns", str(len(runs)) if runs else None)
    out += m("BenchModelSuccess", f"{sum(s['successes'] for s in model)}/{sum(s['runs'] for s in model)}"
             if model else None)
    out += m("BenchRandomSuccess", f"{sum(s['successes'] for s in rnd)}/{sum(s['runs'] for s in rnd)}"
             if rnd else None)
    (gen / "tab_bench.tex").write_text(_bench_table(summary), encoding="utf-8")
    (gen / "macros_bench.tex").write_text(out, encoding="utf-8")


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("root", help="folder holding the run folders (e.g. runs/exp)")
    ap.add_argument("--paper", help="write LaTeX tables into this folder (e.g. paper/generated)")
    ap.add_argument("--smoke", help="smoke-test run folder, for the reference/witness table")
    ap.add_argument("--bench", action="store_true", help="root is a benchmark (runs/bench): write tab_bench.tex")
    a = ap.parse_args(argv)
    runs, summary = aggregate(a.root)
    print(f"{len(runs)} runs, {len(summary)} variants")
    for s in summary:
        print(f"  {s['spec']:<3} {s['variant']:<10} runs {s['runs']:>2}  spec met {s['success_rate'] * 100:5.1f} %  "
              f"median sims to success {s['median_sims_to_success']}")
    if a.paper and a.bench:
        write_bench_tables(a.paper, a.root)
        print(f"benchmark tables written to {a.paper}")
    elif a.paper:
        write_paper_tables(a.paper, a.root, a.smoke)
        print(f"tables written to {a.paper}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile piezo_synth/rawread.py
"""Minimal reader for LTspice .raw waveform files (real-valued transient data, binary or ASCII).

Use uncompressed files (`.options plotwinsize=0`, as reproduce.py --waveforms does).
"""
from __future__ import annotations

import struct
from pathlib import Path
from typing import Dict, List


def _decode_header(raw: bytes):
    """Return (header_text, data_start, kind) with kind 'binary' or 'ascii'."""
    utf16 = len(raw) > 1 and raw[1] == 0
    enc = "utf-16-le" if utf16 else "latin-1"
    for marker, kind in (("Binary:\n", "binary"), ("Values:\n", "ascii")):
        m = marker.encode(enc)
        i = raw.find(m)
        if i < 0 and not utf16:
            m = marker.replace("\n", "\r\n").encode(enc)
            i = raw.find(m)
        if i >= 0:
            return raw[:i].decode(enc), i + len(m), kind, enc
    raise ValueError("not an LTspice raw file: no 'Binary:' or 'Values:' section")


def read_raw(path) -> Dict[str, List[float]]:
    raw = Path(path).read_bytes()
    header, start, kind, enc = _decode_header(raw)
    lines = header.replace("\r\n", "\n").split("\n")
    info, names = {}, []
    in_vars = False
    for line in lines:
        if in_vars:
            parts = line.split()
            if len(parts) >= 3:
                names.append(parts[1])
            continue
        if line.startswith("Variables:"):
            in_vars = True
            continue
        if ":" in line:
            k, v = line.split(":", 1)
            info[k.strip().lower()] = v.strip()
    nvars = int(info["no. variables"])
    npts = int(info["no. points"])
    names = names[:nvars]
    flags = info.get("flags", "").lower()
    if "complex" in flags:
        raise ValueError("complex (AC) data is not supported")
    out: Dict[str, List[float]] = {n: [] for n in names}
    if kind == "binary":
        double = "double" in flags
        fmt = "<" + "d" * nvars if double else "<d" + "f" * (nvars - 1)
        size = struct.calcsize(fmt)
        data = raw[start:]
        npts = min(npts, len(data) // size)
        for k in range(npts):
            vals = struct.unpack_from(fmt, data, k * size)
            for n, v in zip(names, vals):
                out[n].append(v)
    else:
        toks = raw[start:].decode(enc).split()
        step = nvars + 1
        for k in range(min(npts, len(toks) // step)):
            vals = toks[k * step + 1:(k + 1) * step]
            for n, v in zip(names, vals):
                out[n].append(float(v.split(",")[0]))
    if names and names[0].lower() == "time":
        out[names[0]] = [abs(t) for t in out[names[0]]]   # LTspice marks some points with a negative time
    return out


In [ ]:
%%writefile piezo_synth/run.py
"""Entry point for the cluster job.

    python3 -u -m piezo_synth.run --config run_config.json --out runs/<id>    full experiment
    python3 -u -m piezo_synth.run --smoke --out runs/smoke                   reference + witness, no model
    python3 -u -m piezo_synth.run --fake --out runs/dry                      wiring check without LTspice/Gemma
    python3 -u -m piezo_synth.run --baseline random --config c.json --out runs/<id>   model-free baseline
"""
from __future__ import annotations

import argparse
import json
import random
import math
import re
import shutil
import time
from dataclasses import fields
from pathlib import Path

from . import spec
from .circuit import design_hash, to_dict, topology_hash
from .evaluate import Evaluation
from .baselines import RandomProposer
from .llm import GemmaClient, ScriptedLLM
from .netlist import reference_circuit, spec_witness, witness_circuit
from .prompts import condition_table
from .records import Candidate, RunRecorder
from .report import build_report
from .screen import screen
from .search import SearchConfig, evaluate_circuit, run_search
from .sim import Simulator, ltspice_backend


# ---------------------------------------------------------------- dry-run stand-ins

class _FakeResult:
    def __init__(self, meas, folder):
        self.ok, self.meas, self.error, self.folder = True, meas, "", folder


def _uses(text: str, model: str) -> bool:
    return re.search(rf"^[DS]\w* .* {model}$", text, re.M) is not None


def analytic_backend(text: str, folder: str, timeout: float):
    """Plausible, deterministic numbers for a dry run (NOT a circuit simulation)."""
    Path(folder).mkdir(parents=True, exist_ok=True)
    (Path(folder) / "circuit.net").write_text(text, encoding="utf-8")
    ip = float(re.search(r"^Ipz pzn pzp SINE\(0 (\S+) (\S+)\)", text, re.M).group(1))
    if re.search(r"^Vst vsn 0 ", text, re.M):          # screening bench: net power at 1.8 V
        f = float(re.search(r"^Ipz pzn pzp SINE\(0 \S+ (\S+)\)", text, re.M).group(1))
        cp = float(re.search(r"^Cpz pzp pzn (\S+)$", text, re.M).group(1))
        p = (2.5 if _uses(text, "SWI") else 0.5) * ip ** 2 / (4 * math.pi ** 2 * f * cp)
        return _FakeResult({"isn": p * spec.SCREEN["rsense"] / spec.SCREEN["vstore"], "vout": spec.SCREEN["vstore"]}, folder)
    rl = float(re.search(r"^RL out 0 (\S+)$", text, re.M).group(1))
    boost = 2.5 if _uses(text, "SWI") else 1.0
    v = boost * (ip / 50e-6) * 2.2 * min(1.0, rl / 330e3) ** 0.3
    if _uses(text, "DZ33"):
        v = min(v, 3.35)
    meas = {"vavg": v, "vavg_prev": v - 0.005, "vpp": 0.02, "vmax": v * 1.02,
            "pload": v * v / rl, "pext": 1.3 * v * v / rl}
    if v > 1.85:
        meas["tup"] = 0.2
    return _FakeResult(meas, folder)


def _fake_replies() -> list:
    ref = to_dict(reference_circuit())
    clamp = dict(ref, name="bridge + zener clamp", move="modify_topology")
    clamp["components"] = ref["components"] + [{"id": "D5", "type": "D", "nodes": ["0", "out"], "model": "DZ33"}]
    sshi = dict(to_dict(witness_circuit()), name="p-SSHI + bridge + clamp", move="new_topology")
    dup = json.loads(json.dumps(sshi).replace('"n1"', '"mid"'))
    return [json.dumps(clamp), json.dumps(dup), json.dumps(sshi)]


# ---------------------------------------------------------------- smoke test

def screen_sign_suspect(scr: dict) -> bool:
    """True if the screen's sign looks wrong. A small negative net power for the reference on one
    condition is physical (S4: the open-circuit voltage is below 1.8 V plus two diode drops, so the bridge
    delivers nothing and the source loses the diodes' reverse leakage); the witness losing power, or the
    reference losing it on every condition, is not."""
    ref = [v for v in scr["reference"]["p_net"].values() if v is not None]
    wit = [v for v in scr["witness"]["p_net"].values() if v is not None]
    return any(v < 0 for v in wit) or (bool(ref) and all(v <= 0 for v in ref))


def run_smoke(sim: Simulator, rec: RunRecorder) -> None:
    """Reference and witness on all conditions: checks the simulator setup and the spec's difficulty."""
    rec.write_json("spec.json", spec.spec_dict())
    rec.write_json("config.json", {"mode": "smoke"})
    rec.progress(phase="sim", stop_reason=None)
    out = {}
    for k, (circ, origin) in enumerate(((reference_circuit(), "reference"), (spec_witness(), "witness"))):
        cid = f"c{k:04d}"
        for split, conds in (("search", spec.search_conditions()), ("heldout", spec.heldout_conditions())):
            rec.log("sim", f"{origin}: {len(conds)} {split} conditions")
            t = time.time()
            ev = evaluate_circuit(circ, conds, sim, f"{origin}-{split}")
            rec.log("eval", f"{origin} {split}: {ev.cond_pass}/{len(conds)} conditions pass ({time.time() - t:.1f} s)")
            print(condition_table(ev, conds), flush=True)
            rec.candidate(Candidate(cid, 0, origin, None, circ, topology_hash(circ), design_hash(circ), ev,
                                    origin, circ.name, len(circ.components)), split=split)
            out[(origin, split)] = ev
    ref_s1 = out[("reference", "search")].meas.get("S1", {})
    pext = ref_s1.get("pext")
    if pext is None or pext <= 0:
        rec.log("eval", f"WARNING: reference P_ext on S1 is {pext}; expected > 0. Check the sign of the pext "
                        f".meas in netlist.py before trusting P_ext columns.")
    wit_tup = [m.get("tup") for m in out[("witness", "search")].meas.values() if m.get("tup") is not None]
    if wit_tup and all(abs(t - spec.vmin()) < 1e-9 for t in wit_tup):
        rec.log("eval", f"WARNING: every t_up equals {spec.vmin():g}: ltspice_run is reading the threshold, not the crossing "
                        "time, from the 'tup' measurement line. Fix its parsing before the full run.")
    # the power screen on both circuits, with their own values: checks its sign and its calibration
    scr = {}
    for origin, circ in (("reference", reference_circuit()), ("witness", spec_witness())):
        r = screen(circ, sim, random.Random(1), f"{origin}-screen", n_presets=1)
        scr[origin] = {"passed": r.passed, "worst_ratio": r.worst_ratio, "ratio": r.ratio, "gain": r.gain,
                       "p_net": r.p_net, "p_req": r.p_req}
        rec.log("screen", f"{origin}: net power at {spec.SCREEN['vstore']:g} V vs need: {r.summary()}; "
                          f"gain over ideal FBR: " + ", ".join(f"{k} {v:.2f}" for k, v in r.gain.items() if v is not None))
    rec.write_json("smoke_screen.json", scr)
    if screen_sign_suspect(scr):
        rec.log("screen", "WARNING: net powers look sign-flipped (the witness loses power, or the reference loses "
                          "it everywhere): check the 'isn' measurement (V(out,vsn)) before trusting the screen.")
    if not scr["witness"]["passed"]:
        rec.log("screen", f"WARNING: the witness fails the screen (worst {scr['witness']['worst_ratio'] * 100:.0f} %); "
                          f"consider lowering screen_alpha.")
    ref_fail = [c for c in spec.search_conditions()
                if not all(ch.passed for ch in out[("reference", "search")].checks if ch.condition == c.id)]
    rec.log("eval", f"reference fails {len(ref_fail)}/{len(spec.search_conditions())} search conditions: {', '.join(c.id for c in ref_fail)} "
                    f"(the spec is meant to make it fail several)")
    st = sim.stats
    rec.timing({"wall_s": round(rec.elapsed(), 1), "model_s": 0, "model_calls": 0, "prompt_tokens": 0,
                "completion_tokens": 0, "reasoning_tokens": 0, "sim_seconds": round(st["sim_seconds"], 1),
                "sim_wall_s": round(st["wall_seconds"], 1), "n_sims": st["n_sims"], "cache_hits": st["cache_hits"],
                "refine_s": 0})
    rec.write_json("outcome.json", {"stop_reason": "smoke", "met_search": out[("witness", "search")].all_pass,
                                    "met_all": out[("witness", "search")].all_pass and out[("witness", "heldout")].all_pass,
                                    "selected_id": "c0001", "selected_name": spec_witness().name,
                                    "selected_origin": "witness", "reference_id": "c0000", "failed_checks": []})
    rec.progress(phase="done", stop_reason="smoke")


# ---------------------------------------------------------------- main

def load_config(path) -> SearchConfig:
    if not path:
        return SearchConfig()
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    known = {f.name for f in fields(SearchConfig)}
    unknown = sorted(set(raw) - known)
    if unknown:
        print(f"WARNING: ignoring unknown config keys: {', '.join(unknown)}", flush=True)
    return SearchConfig(**{k: v for k, v in raw.items() if k in known})


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default=f"runs/{time.strftime('%Y%m%d-%H%M%S')}")
    ap.add_argument("--config")
    ap.add_argument("--smoke", action="store_true", help="reference and witness only, no model")
    ap.add_argument("--fake", action="store_true", help="scripted model + analytic stand-in for LTspice")
    ap.add_argument("--baseline", choices=["random"], help="replace the model by a model-free proposer")
    ap.add_argument("--label", default="", help="variant name recorded in run_meta.json (for experiments.py)")
    ap.add_argument("--seed", type=int, help="overrides the config's seed")
    ap.add_argument("--spec", default="P1", choices=list(spec.SPECS), help="which specification (spec.SPECS)")
    ap.add_argument("--workers", type=int, default=8, help="parallel LTspice simulations")
    ap.add_argument("--sim-timeout", type=float, default=120.0, help="seconds per simulation")
    a = ap.parse_args(argv)
    with spec.using(a.spec):
        return _main(a, argv)


def _main(a, argv) -> int:
    out = Path(a.out)
    cfg = load_config(a.config)
    if a.seed is not None:
        cfg.seed = a.seed
    if a.fake:
        cfg.heartbeat_s = min(cfg.heartbeat_s, 1.0)
    backend = analytic_backend if a.fake else ltspice_backend()
    sim = Simulator(backend, out, workers=a.workers, timeout=a.sim_timeout)
    rec = RunRecorder(out)
    rec.write_json("run_meta.json", {"label": a.label, "baseline": a.baseline, "fake": a.fake, "smoke": a.smoke,
                                     "seed": cfg.seed, "spec": a.spec, "argv": argv})
    rec.log("setup", f"output in {out}; mode {'smoke' if a.smoke else 'search'}{' (fake)' if a.fake else ''}; spec {a.spec} ({spec.SPECS[a.spec].title})"
                     f"{' (baseline ' + a.baseline + ')' if a.baseline else ''}; "
                     f"{a.workers} parallel simulations")
    free = shutil.disk_usage(out).free / 2**30
    rec.log("setup", f"free disk where results go: {free:.1f} GiB")
    if free < 2:
        rec.log("setup", "WARNING: less than 2 GiB free; a full disk makes simulations fail. Free space first, "
                         "e.g. python3 tools/clean_raw.py runs --delete")
    try:
        if a.smoke:
            run_smoke(sim, rec)
        else:
            if a.baseline == "random":
                llm = RandomProposer(seed=cfg.seed, per_round=cfg.proposals_per_round)
            else:
                llm = ScriptedLLM(_fake_replies()) if a.fake else GemmaClient()
            run_search(cfg, llm, sim, rec)
    finally:
        build_report(out)
        rec.log("done", f"report: {out / 'report.md'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile piezo_synth/reproduce.py
"""Re-simulate a saved circuit on chosen conditions, optionally keeping full waveforms.

    python3 -m piezo_synth.reproduce runs/<id>/selected/circuit.json --out runs/<id>/repro \
        --conditions all --waveforms
"""
from __future__ import annotations

import argparse
import json
from dataclasses import asdict
from pathlib import Path

from . import spec
from .circuit import parse_circuit
from .evaluate import evaluate
from .netlist import build_netlist
from .prompts import condition_table
from .sim import Simulator, ltspice_backend


def find_raw(folder):
    """The transient waveform file of a simulation folder (searched in subfolders too; .op.raw skipped)."""
    folder = Path(folder)
    if not folder.exists():
        return None
    raws = sorted(r for r in folder.rglob("*.raw") if not r.name.endswith(".op.raw"))
    return str(raws[0]) if raws else None


def pick_conditions(arg: str):
    if arg == "search":
        return spec.search_conditions()
    if arg == "heldout":
        return spec.heldout_conditions()
    if arg == "all":
        return list(spec.CONDITIONS)
    return [spec.condition(c.strip()) for c in arg.split(",") if c.strip()]


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("circuit", help="circuit.json (e.g. runs/<id>/selected/circuit.json)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--conditions", default="all", help="search | heldout | all | comma list such as S1,H2")
    ap.add_argument("--spec", default=None, help="specification (default: the run's spec.json, else P1)")
    ap.add_argument("--waveforms", action="store_true", help="save all signals, uncompressed .raw")
    ap.add_argument("--fake", action="store_true", help="analytic stand-in for LTspice (dry run)")
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args(argv)
    name = a.spec or _run_spec(Path(a.circuit)) or "P1"
    with spec.using(name):
        print(f"spec {name}: {spec.SPECS[name].title}", flush=True)
        return _main(a)


def _run_spec(circuit_path: Path):
    """The spec recorded in the run folder that holds this circuit, if any."""
    for d in circuit_path.resolve().parents:
        p = d / "spec.json"
        if p.exists():
            try:
                return json.loads(p.read_text(encoding="utf-8")).get("name")
            except ValueError:
                return None
    return None


def _main(a) -> int:
    circ = parse_circuit(json.loads(Path(a.circuit).read_text(encoding="utf-8")))
    conds = pick_conditions(a.conditions)
    if a.fake:
        from .run import analytic_backend
        backend = analytic_backend
    else:
        backend = ltspice_backend()
    sim = Simulator(backend, Path(a.out), workers=a.workers, keep_raw=a.waveforms)
    jobs = []
    for cond in conds:
        text = build_netlist(circ, cond, title=f"reproduce {circ.name}")   # only the signals the plots use
        if a.waveforms:
            text = text.replace("\n.end\n", "\n.options plotwinsize=0\n.end\n")
        jobs.append((cond.id, text))
    results = sim.run(jobs)
    ev = evaluate({c.id: r for c, r in zip(conds, results)}, conds)
    print(condition_table(ev, conds), flush=True)
    waves = {}
    for cond, r in zip(conds, results):
        waves[cond.id] = find_raw(r.folder)
        if a.waveforms and waves[cond.id] is None:
            folder = Path(r.folder)
            listing = sorted(str(x.relative_to(folder)) for x in folder.rglob("*"))[:20] if folder.exists() else []
            print(f"{cond.id}: WARNING no .raw waveform file in {r.folder}; it contains: {listing}")
        if not r.ok:
            print(f"{cond.id}: simulation failed: {r.error[:300]}")
    (Path(a.out) / "reproduce.json").write_text(json.dumps({
        "circuit": a.circuit, "conditions": [c.id for c in conds], "all_pass": ev.all_pass,
        "checks": [asdict(c) for c in ev.checks], "meas": ev.meas, "raw_files": waves}, indent=1), encoding="utf-8")
    print(f"all pass: {ev.all_pass}; details in {Path(a.out) / 'reproduce.json'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile piezo_synth/speccheck.py
"""Difficulty and feasibility check of every specification, without the language model.

    python3 -u -m piezo_synth.speccheck --out runs/speccheck            all specs (a cluster job)
    python3 -u -m piezo_synth.speccheck --specs P2,P3 --budget 96 --out runs/speccheck
    python3 -m piezo_synth.speccheck --fake --out /tmp/sc                 wiring check

For each spec it simulates, on the search and held-out conditions:
  reference  the full-bridge rectifier (expected to fail);
  textbook   the P1 witness, parallel SSHI + bridge + 3.3 V zener clamp, with its own values and after
             sizing by the same evolution strategy the search uses: the textbook answer at its best;
  witness    the spec's own hand-made circuit (netlist.spec_witness), also sized.
The verdict: "textbook_solves" (the spec does not need anything beyond the textbook circuit),
"witness_solves" (the textbook fails but a known-feasible design exists), or "unproven" (neither met it:
the spec may be infeasible with this vocabulary).
"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path
from typing import List

from . import spec
from .bench import evaluate_circuit, evaluate_many
from .circuit import Circuit, to_dict
from .netlist import reference_circuit, spec_witness, witness_circuit
from .screen import screen
from .sim import Simulator, ltspice_backend
from .sizer import Sizer


def verdict(textbook_met: bool, witness_met: bool) -> str:
    if textbook_met:
        return "textbook_solves"
    return "witness_solves" if witness_met else "unproven"


def _size(c: Circuit, sim, budget: int, seed: int, label: str):
    conds = spec.search_conditions()
    ev = evaluate_circuit(c, conds, sim, f"{label}-proposed")
    sz = Sizer(c, ev, budget, random.Random(seed))
    k = 0
    while not sz.done:
        batch = sz.ask()
        if not batch:
            break
        evs = evaluate_many(batch, conds, sim, [f"{label}-g{k}-{i}" for i in range(len(batch))])
        sz.tell(batch, evs)
        k += 1
    return ev, sz


def _entry(c: Circuit, sim, budget: int, seed: int, label: str, size: bool) -> dict:
    held = spec.heldout_conditions()
    ev0 = evaluate_circuit(c, spec.search_conditions(), sim, f"{label}-search")
    d = {"name": c.name, "search_pass": ev0.cond_pass, "search_met": ev0.all_pass,
         "failed": sorted({ch.condition + ":" + ch.criterion for ch in ev0.checks if not ch.passed})}
    try:
        r = screen(c, sim, random.Random(seed), f"{label}-screen", n_presets=1)
        d["screen_ratio"] = r.ratio
    except Exception as e:  # noqa: BLE001 - a screen failure must not stop the check of the other specs
        d["screen_error"] = str(e)[:200]
    best = c
    if size:
        _, sz = _size(c, sim, budget, seed, label)
        best = sz.best
        d.update({"sized_search_pass": sz.best_ev.cond_pass, "sized_search_met": sz.best_ev.all_pass,
                  "sized_evals": sz.n_evals, "sizer_stop": sz.reason, "sized_circuit": to_dict(best),
                  "sized_failed": sorted({ch.condition + ":" + ch.criterion for ch in sz.best_ev.checks
                                          if not ch.passed})})
    evh = evaluate_circuit(best, held, sim, f"{label}-heldout")
    d.update({"heldout_pass": evh.cond_pass, "heldout_n": len(held)})
    return d


def check(name: str, root: Path, backend, budget: int, workers: int, seed: int) -> dict:
    with spec.using(name):
        out = root / name
        sim = Simulator(backend, out, workers=workers)
        t = time.time()
        print(f"[spec] {name}: {spec.SPECS[name].title}", flush=True)
        res = {"spec": name, "title": spec.SPECS[name].title, "hardness": spec.SPECS[name].hardness}
        res["reference"] = _entry(reference_circuit(), sim, budget, seed, "reference", size=False)
        res["textbook"] = _entry(witness_circuit(), sim, budget, seed, "textbook", size=True)
        w = spec_witness()
        if w.name != witness_circuit().name:
            res["witness"] = _entry(w, sim, budget, seed, "witness", size=True)
        else:
            res["witness"] = {"name": "(same as textbook)", **{k: v for k, v in res["textbook"].items() if k != "name"}}
        res["verdict"] = verdict(bool(res["textbook"].get("sized_search_met")),
                                 bool(res["witness"].get("sized_search_met")))
        res["n_sims"] = sim.stats["n_sims"]
        res["wall_s"] = round(time.time() - t, 1)
        out.mkdir(parents=True, exist_ok=True)
        (out / "summary.json").write_text(json.dumps(res, indent=1), encoding="utf-8")
        tb, wi = res["textbook"], res["witness"]
        print(f"[spec] {name}: reference {res['reference']['search_pass']}/6; textbook {tb['search_pass']}/6 "
              f"-> sized {tb.get('sized_search_pass')}/6; witness sized {wi.get('sized_search_pass')}/6; "
              f"verdict {res['verdict']} ({res['n_sims']} sims, {res['wall_s']:.0f} s)", flush=True)
        return res


def _table(rows: List[dict]) -> str:
    lines = ["| spec | title | reference | textbook as given | textbook sized | witness sized | held-out (sized) "
             "| verdict |", "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        tb, wi = r["textbook"], r["witness"]
        best = wi if wi.get("sized_search_met") and not tb.get("sized_search_met") else tb
        lines.append(f"| {r['spec']} | {r['title']} | {r['reference']['search_pass']}/6 | {tb['search_pass']}/6 | "
                     f"{tb.get('sized_search_pass')}/6 | {wi.get('sized_search_pass')}/6 | "
                     f"{best.get('heldout_pass')}/{best.get('heldout_n')} | {r['verdict']} |")
    return "\n".join(lines) + "\n"


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default="runs/speccheck")
    ap.add_argument("--specs", default=",".join(spec.SPECS))
    ap.add_argument("--budget", type=int, default=96, help="sizing evaluations per circuit")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--fake", action="store_true", help="analytic stand-in for LTspice (NOT a simulation)")
    a = ap.parse_args(argv)
    root = Path(a.out)
    if a.fake:
        from .run import analytic_backend
        backend = analytic_backend
    else:
        backend = ltspice_backend()
    names = [n.strip() for n in a.specs.split(",") if n.strip()]
    rows = [check(n, root, backend, a.budget, a.workers, a.seed) for n in names]
    root.mkdir(parents=True, exist_ok=True)
    (root / "summary.json").write_text(json.dumps(rows, indent=1), encoding="utf-8")
    (root / "speccheck.md").write_text("# Specification check\n\n" + _table(rows), encoding="utf-8")
    print(_table(rows), flush=True)
    print("[done] spec check written to", root / "speccheck.md", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 3. Settings

- `max_rounds`: model calls at most (each ~1-4 min with thinking on); `proposals_per_round`: designs per call.
- `min_rounds`: the model may only declare itself done (and stagnation may only stop the run) after this many.
- `wall_clock_s`, `max_sims`: time and simulation budgets (0 = no simulation budget).
- `stagnation`: stop after this many rounds without improving the best proposed design.
- `screen`, `screen_presets`, `screen_alpha`: the power screen (alpha = fraction of the load's need required).
- `sizer` (`es`, `coord` or `none`), `size_budget`, `size_lambda`: value sizing.
- `diagnostics`: plain-language diagnoses in the feedback.
- `mc_samples`: Monte Carlo samples for the selected design.
- `think`: Gemma reasoning on (recommended); `seed`: random seed for screen presets and sizing.

In [ ]:
CONFIG = dict(max_rounds=12, min_rounds=3, proposals_per_round=3, wall_clock_s=3600, max_sims=3000,
              stagnation=5, screen=True, screen_presets=4, screen_alpha=0.5, sizer="es", size_budget=48,
              size_lambda=8, diagnostics=True, mc_samples=16, think=True, seed=1)
Path("run_config.json").write_text(json.dumps(CONFIG, indent=1))
CONFIG

## 4. Smoke test (first run only)

Expect: 20 simulations, `[sim]`/`[eval]` lines and two tables per circuit. Check that the reference
**fails** several search conditions (that is what makes the spec challenging) and that the witness
passes more of them. If the job prints a WARNING about `P_ext`, read the README section "First run".

In [ ]:
smoke = run("bash -o pipefail -c 'rm -rf runs/smoke && mkdir -p runs && python3 -u -m piezo_synth.run --smoke --out runs/smoke "
            "2>&1 | tee runs/smoke.log'",
            image=LTSPICE, gpus=0, cpus=8, memory="16Gi", timeout=3600, name="piezo-smoke")
log(smoke)

## 5. Single run

One CPU job runs everything: Gemma calls (served on the cluster's GPUs), LTspice simulations in 8
parallel threads, value tuning while Gemma thinks, the held-out check and the report. The console
output is also saved to `runs/<RUN_ID>.log` (the job's own log is deleted after 48 hours).

In [ ]:
job = run(f"bash -o pipefail -c 'mkdir -p runs && python3 -u -m piezo_synth.run --config run_config.json "
          f"--out runs/{RUN_ID} 2>&1 | tee runs/{RUN_ID}.log'",
          image=LTSPICE, gpus=0, cpus=8, memory="16Gi", timeout=4 * 3600, name="piezo-synth")
print("submitted", job, "-> runs/" + RUN_ID)
log(job)

## 6. Progress, results, figures

To look at an earlier run, set `RUN_ID` to its folder name under `runs/` first.

In [ ]:
RUN = Path("runs") / RUN_ID
p = json.loads((RUN / "progress.json").read_text()) if (RUN / "progress.json").exists() else {}
print(f"phase: {p.get('phase')}   round {p.get('iteration')}/{p.get('max_rounds')}   "
      f"simulations {p.get('n_sims')}   elapsed {p.get('elapsed_s', 0) / 60:.1f} min")
print(f"best so far: {p.get('best_id')} {p.get('best_name')!r}  score {p.get('best_scalar')}  "
      f"conditions passed {p.get('best_cond_pass')}/6")
if p.get("phase") == "model":
    print(f"waiting for the model for {p.get('model_wait_s', 0)} s")
if p.get("stop_reason"):
    print("stop reason:", p["stop_reason"])

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from piezo_synth.report import build_report

build_report(RUN)          # refresh (also works on an unfinished or killed run)
display(Markdown((RUN / "report.md").read_text()))
cands = pd.read_csv(RUN / "candidates.csv")
meas = pd.read_csv(RUN / "measurements.csv")
checks = pd.read_csv(RUN / "checks.csv")
summary = json.loads((RUN / "summary.json").read_text())
SEL, REF = summary["selected_id"], summary["reference_id"]
cands.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from piezo_synth import spec

figdir = RUN / "figures"; figdir.mkdir(exist_ok=True)
search = cands[cands.split == "search"].copy()

# 1. score over the search, proposals coloured by move, sizing steps in grey
fig, ax = plt.subplots(figsize=(8, 4))
search["order"] = range(len(search))
best = search.scalar.cummax()
ax.plot(search.order, best, "k-", lw=1.5, label="best so far")
colours = {"new_topology": "tab:blue", "modify_topology": "tab:orange", "resize": "tab:green",
           "size": "lightgrey", "refine": "lightgrey", "audit": "wheat", "reference": "tab:red"}
for mv, grp in search.groupby("move"):
    small = mv in ("size", "refine", "audit")
    ax.scatter(grp.order, grp.scalar, s=10 if small else 25, c=colours.get(mv, "k"), label=mv, zorder=2 if small else 3)
ax.set(xlabel="design evaluated (in order)", ylabel="score", title="Search progress")
ax.legend(fontsize=8, ncol=3); fig.tight_layout()
fig.savefig(figdir / "search_progress.png", dpi=200); fig.savefig(figdir / "search_progress.pdf")

# 2. pass/fail map: selected vs reference, all conditions x criteria
conds = [c.id for c in spec.CONDITIONS]; crits = [c.id for c in spec.CRITERIA]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (cid, title) in zip(axes, [(SEL, f"selected ({SEL})"), (REF, f"reference ({REF})")]):
    sub = checks[checks.candidate == cid]
    grid = np.full((len(crits), len(conds)), np.nan)
    for _, r in sub.iterrows():
        grid[crits.index(r.criterion), conds.index(r.condition)] = 1.0 if str(r.passed) == "True" else 0.0
    ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(conds)), conds); ax.set_yticks(range(len(crits)), crits); ax.set_title(title)
    ax.axvline(5.5, color="k", lw=1)          # left: search conditions, right: held-out
fig.suptitle("green = pass, red = fail; conditions right of the line are held out"); fig.tight_layout()
fig.savefig(figdir / "pass_fail.png", dpi=200); fig.savefig(figdir / "pass_fail.pdf")

# 3. output voltage and load power per condition
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.arange(len(conds)); w = 0.38
for k, (cid, lab) in enumerate([(SEL, "selected"), (REF, "reference")]):
    sub = meas[meas.candidate == cid].set_index("condition").reindex(conds)
    axes[0].bar(x + (k - 0.5) * w, sub.vavg, w, label=lab)
    axes[1].bar(x + (k - 0.5) * w, sub.pload * 1e6, w, label=lab)
for v in (1.8, 3.6):
    axes[0].axhline(v, color="k", ls="--", lw=0.8)
axes[0].set(ylabel="mean V(out), V", xticks=x, xticklabels=conds)
axes[1].set(ylabel="load power, uW", xticks=x, xticklabels=conds)
axes[0].legend(); fig.tight_layout()
fig.savefig(figdir / "vout_power.png", dpi=200); fig.savefig(figdir / "vout_power.pdf")
print("figures saved in", figdir)

### Waveforms

Re-simulates the selected design and the reference on S1 and S4 with all signals saved, then plots
V(out). This is also how to reproduce any design: point `reproduce` at its `circuit.json`.

In [ ]:
wave = run(f"python3 -u -m piezo_synth.reproduce runs/{RUN_ID}/selected/circuit.json "
           f"--out runs/{RUN_ID}/waves-selected --conditions S1,S4 --waveforms && "
           f"python3 -u -m piezo_synth.reproduce runs/{RUN_ID}/candidates/{REF}/circuit.json "
           f"--out runs/{RUN_ID}/waves-reference --conditions S1,S4 --waveforms",
           image=LTSPICE, gpus=0, cpus=4, memory="8Gi", timeout=1800, name="piezo-waves")
log(wave)

In [ ]:
from piezo_synth.rawread import read_raw

missing = []
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, cond in zip(axes, ["S1", "S4"]):
    for lab in ("selected", "reference"):
        info_file = RUN / f"waves-{lab}" / "reproduce.json"
        if not info_file.exists():
            missing.append(f"{lab}: {info_file} not found (has the Waveforms job finished? see log(wave))")
            continue
        raw = json.loads(info_file.read_text())["raw_files"].get(cond)
        if not raw:
            missing.append(f"{lab} {cond}: no .raw file was produced (see the WARNING lines in log(wave))")
            continue
        d = read_raw(raw)
        key = next((k for k in d if k.lower() == "v(out)"), None)
        if key is None:
            missing.append(f"{lab} {cond}: V(out) not in {raw}; signals: {list(d)[:8]}")
            continue
        ax.plot(d["time"], d[key], lw=0.8, label=lab)
    for v in (1.8, 3.6):
        ax.axhline(v, color="k", ls="--", lw=0.6)
    ax.set(title=cond, xlabel="time, s", ylabel="V(out), V")
    if ax.get_legend_handles_labels()[0]:
        ax.legend()
fig.tight_layout(); fig.savefig(figdir / "waveforms.png", dpi=200); fig.savefig(figdir / "waveforms.pdf")
for m in missing:
    print("MISSING:", m)

## 7. Experiment campaign (for the paper)

Every variant runs the same seeds with the same simulation budget (`max_sims`), so variants are
compared by **simulations used**, which does not depend on how busy the shared Gemma server is.

| variant | what changes |
|---|---|
| `full` | the method as described above |
| `no_screen` | no power screen: every structure is evaluated in full and sized |
| `no_diag` | no plain-language diagnoses in the feedback (tables only) |
| `single` | one design per round instead of three |
| `coord` | the earlier coordinate-search tuner instead of the evolution strategy |
| `no_think` | Gemma's reasoning turned off |
| `random` | no model: random valid circuits through the same screen and sizing |
| `audit` | like `full`, but screen-rejected structures are also sized (measures how often the screen is wrong) |

One cluster job per variant runs its seeds one after another. With 5 seeds a variant takes up to
about 5 hours; all variants run at the same time. Re-running a variant skips seeds already finished.

In [ ]:
from piezo_synth.experiments import CAMPAIGN_BASE, VARIANTS, campaign_loop, write_campaign_configs

SEEDS = [1, 2, 3, 4, 5]
write_campaign_configs("runs/exp", base=dict(CONFIG, **CAMPAIGN_BASE))
campaign = {}
for name in VARIANTS:
    campaign[name] = run(f"bash -c '{campaign_loop(name, SEEDS)}'", image=LTSPICE, gpus=0, cpus=8,
                         memory="16Gi", timeout=12 * 3600, name=f"piezo-{name.replace('_', '-')}")
    print("submitted", name, campaign[name])

In [ ]:
# how far each variant has got (safe to run any time)
for name in VARIANTS:
    done = sorted(glob.glob(f"runs/exp/{name}/seed*/outcome.json"))
    print(f"{name:<10} {len(done)}/{len(SEEDS)} seeds finished")

### Aggregate and write the paper's tables

Writes `runs/exp/experiments.csv` (one row per run), `runs/exp/experiments_summary.csv` (one row per
variant) and the LaTeX tables in `paper_generated/`. Download that folder into the repository's
`paper/generated/` and compile `paper/main.tex`; every number in the paper's results comes from it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from piezo_synth.experiments import aggregate, write_paper_tables

runs_, summary_ = aggregate("runs/exp")
write_paper_tables("paper_generated", "runs/exp", smoke="runs/smoke")
df = pd.DataFrame(runs_); sm = pd.DataFrame(summary_)
display(sm)
figdir = Path("paper_generated"); figdir.mkdir(exist_ok=True)
order = [v for v in ["full", "no_screen", "no_diag", "single", "coord", "no_think", "random", "audit"] if v in set(df.variant)]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(order, [sm.set_index("variant").success_rate.get(v, 0) * 100 for v in order], color="tab:blue")
axes[0].set(ylabel="runs meeting the spec (%)", ylim=(0, 100))
data = [df[(df.variant == v) & df.met_search].first_met_sims.dropna().values for v in order]
axes[1].boxplot(data, labels=order); axes[1].set(ylabel="simulations to first success")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout(); fig.savefig(figdir / "fig_variants.pdf"); fig.savefig(figdir / "fig_variants.png", dpi=200)
# search progress of the first successful full run, for the paper
ok = df[(df.variant == "full") & df.met_search]
if len(ok):
    run_dir = Path(ok.iloc[0].run)
    c = pd.read_csv(run_dir / "candidates.csv"); c = c[c.split == "search"].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(c.index, c.scalar.cummax(), "k-", lw=1.2, label="best so far")
    for mv, g in c.groupby("move"):
        ax.scatter(g.index, g.scalar, s=6 if mv in ("size", "audit") else 18, label=mv)
    ax.set(xlabel="design evaluated", ylabel="score"); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); fig.savefig(figdir / "fig_progress.pdf")
import shutil; shutil.make_archive("paper_generated", "zip", "paper_generated")
print("download paper_generated.zip and unpack it into paper/generated/")